# Assessment 1: Empirical Assignment
### 25882 AI-powered Investment and Risk Management

**Group members (name, student ID):**
- TODO: Name 1, Student ID 1
- TODO: Name 2, Student ID 2 *(delete this line if working alone)*

## Part B choices

We attempt two extensions. **B1 (Portfolio optimisation)**, from Category B, is
Extension 1. **A3 (Tail risk and stress testing)**, from Category A, is Extension 2.
This satisfies your requirement that the two options come from different categories,
with at least one from Category A or B. Both of ours qualify.

The two extensions are deliberately linked, and we want you to see that link clearly.
Extension 1 builds minimum-variance, maximum-Sharpe and risk-parity portfolios.
Extension 2 stress-tests those same portfolios against the fat-tailed,
correlation-spiking behaviour already documented in A3. It also compares an EVT-based
tail estimate directly against the historical and parametric VaR/ES computed there.

One labelling note for you. The assignment's Category A option 3 is also called "A3."
That name happens to collide with this notebook's own core section A3 (Baseline risk
report). Per your own heading convention, we head Part B options `## Extension 1` and
`## Extension 2`, not a reused `## A3`. So there is no real heading collision, just a
naming coincidence we want to flag so you don't read it as an error.

In [ ]:
import importlib.util
import subprocess
import sys

def _ensure_installed(packages):
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        print(f"Installing missing packages into this kernel: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

_ensure_installed(["numpy", "pandas", "yfinance"])

import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data_cache")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"Notebook environment set up. Random seed fixed at {RANDOM_SEED}.")

## A1: Universe selection and data acquisition

### Asset universe and justification

| Ticker | Name | Sector | Exchange | Currency |
|---|---|---|---|---|
| AAPL | Apple Inc. | Information Technology | NASDAQ | USD |
| JPM | JPMorgan Chase & Co. | Financials | NYSE | USD |
| XOM | ExxonMobil Corp. | Energy | NYSE | USD |
| JNJ | Johnson & Johnson | Health Care | NYSE | USD |
| 7203.T | Toyota Motor Corp. | Consumer Discretionary (Automobiles) | Tokyo Stock Exchange | JPY |

Here is why we chose this universe. We picked four large-cap US stocks spanning four
distinct GICS sectors: technology, financials, energy, and healthcare. We added one
Japan-listed stock, Toyota (7203.T), to satisfy two goals at once.

First, four distinct sectors avoid a degenerate, near-perfectly-correlated universe. A
single-sector selection would flatten the diversification and optimisation results
you'll see in Part B, and we wanted you to see real diversification effects, not an
artefact of picking similar companies.

Second, Toyota trades on the Tokyo Stock Exchange in JPY. That gives us a genuine
cross-currency, cross-trading-calendar asset, which your brief requires, and which
makes the currency-alignment and holiday-calendar corrections in Section A2 actually
mean something. If we had picked five US large-caps in USD instead, we would never have
observed an FX or calendar-alignment effect at all, and we would have understated how
much diversification the equal-weight benchmark in A4 can actually deliver.

We fixed the sample window to 2016-01-01 through 2025-01-01 as an explicit, hard-coded
range rather than "today," so the notebook asks for the exact same data on every
re-run, regardless of when it happens to execute. That's just over nine years, safely
above your 8-year minimum. We also record the date we first downloaded this data
successfully, since prices from public APIs change: vendors restate history and adjust
for corporate actions retroactively, so the download date explains any difference
between our numbers and yours on a later re-run.

In [ ]:
TICKERS = ["AAPL", "JPM", "XOM", "JNJ", "7203.T"]

PRICE_START = "2016-01-01"
PRICE_END = "2025-01-01"

DOWNLOAD_DATE_RECORDED = "2026-09-15"

print(f"Universe: {TICKERS}")
print(f"Requested date range: {PRICE_START} to {PRICE_END}")

In [ ]:
def load_price_panel(tickers, start, end, cache_dir=DATA_DIR, force_download=False):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    raw_path = cache_dir / "raw_close.csv"
    adj_path = cache_dir / "adj_close.csv"

    if not force_download and raw_path.exists() and adj_path.exists():
        raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
        adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
        cached_tickers, requested_tickers = set(raw_close.columns), set(tickers)
        if cached_tickers == requested_tickers:
            log = {
                "source": "local cache",
                "path": str(cache_dir),
                "tickers": list(raw_close.columns),
            }
            print(f"Loaded cached panel from {cache_dir}/ "
                  f"(delete raw_close.csv / adj_close.csv there to force a fresh download).")
            return raw_close, adj_close, log
        else:

            print(f"Cache at {cache_dir}/ contains {sorted(cached_tickers)}, which does not "
                  f"match the requested tickers {sorted(requested_tickers)} -- ignoring this "
                  "stale/mismatched cache and attempting a fresh download instead.")

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame for every ticker")

        raw_cols, adj_cols = {}, {}
        if isinstance(data.columns, pd.MultiIndex):
            for t in tickers:
                try:
                    sub = data[t]
                except KeyError:
                    print(f"  WARNING: no data at all returned for {t}; dropping from panel")
                    continue
                if sub["Close"].dropna().empty:
                    print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
                    continue
                raw_cols[t] = sub["Close"]
                adj_cols[t] = sub["Adj Close"]
        else:

            if len(tickers) != 1:
                raise ValueError(
                    f"Expected MultiIndex columns for {len(tickers)} tickers, got flat "
                    f"columns {list(data.columns)}"
                )
            t = tickers[0]
            if data["Close"].dropna().empty:
                print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
            else:
                raw_cols[t] = data["Close"]
                adj_cols[t] = data["Adj Close"]

        if not raw_cols:
            raise ValueError("Every requested ticker came back empty or all-NaN")

        raw_close = pd.DataFrame(raw_cols).sort_index()
        adj_close = pd.DataFrame(adj_cols).sort_index()

        raw_close.to_csv(raw_path)
        adj_close.to_csv(adj_path)

        log = {
            "source": "yfinance (live download)",
            "download_timestamp": datetime.now().isoformat(timespec="seconds"),
            "yfinance_version": getattr(yf, "__version__", "unknown"),
            "requested_tickers": list(tickers),
            "returned_tickers": list(raw_close.columns),
            "requested_start": start,
            "requested_end": end,
        }
        print(f"Downloaded fresh data and cached it to {cache_dir}/.")
        return raw_close, adj_close, log

    except Exception as exc:
        print(f"Live download failed ({exc!r}).")
        if raw_path.exists() and adj_path.exists():
            print("Falling back to the previously cached local files so the "
                  "notebook can still run end to end.")
            raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
            adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
            log = {
                "source": "local cache (after a failed live download)",
                "path": str(cache_dir),
            }
            return raw_close, adj_close, log
        raise RuntimeError(
            f"No live data available and no local cache found at {cache_dir}/. "
            "Cannot proceed -- see the note on reproducibility in the assignment brief."
        ) from exc

**A note on how this loader is built, since it does real work for you.** We call
`yf.download(..., auto_adjust=False)` explicitly. By default, yfinance now sets
`auto_adjust=True` and returns only an adjusted close, discarding the raw price
entirely. Section A2 needs both series, so we turn that default off. We also handle two
different column shapes: yfinance returns MultiIndex (ticker, field) columns for a
multi-ticker request, but sometimes returns flat columns for a single-ticker request
even with `group_by="ticker"`. We handle both shapes here, since we reuse this same
function later for the single-ticker FX and risk-free downloads in A2.

If a ticker returns nothing, or returns an all-NaN Close column, we drop it and report
that we did, before any row-wise NaN handling. Your brief warns about exactly this: drop
empty and all-NaN columns first, then handle rows, or one broken ticker can silently
delete your entire sample.

On success, we cache both wide panels to `cache_dir` as CSV files. On a later call, or a
later notebook re-run, with the cache already present, the function reads the cache
directly and never touches the network. We also check the cached columns against the
tickers currently requested, and treat a mismatch as a cache miss rather than trusting
stale data blindly, since a shared cache directory between two notebook variants could
otherwise hand back the wrong asset's data with no error at all.

If a live download raises, a network error, a rate limit, a vendor outage, which your
brief notes does happen, the function falls back to the local cache if one exists, so
the notebook degrades gracefully instead of crashing on a clean re-run at marking time.

One more function below, `load_ohlcv_preview()`, downloads the full OHLCV panel purely
so you can see what the vendor actually served. It's display-only. Every later section
works from this loader's `raw_close`/`adj_close` instead, which deliberately keep only
Close and Adj Close, since A2 is specifically about auditing those two series.

In [ ]:
raw_close, adj_close, download_log = load_price_panel(TICKERS, PRICE_START, PRICE_END)

print("\nDownload log:")
for k, v in download_log.items():
    print(f"  {k}: {v}")

In [ ]:
def summarize_panel(price_df, label="", min_years=8):
    summary = pd.DataFrame({
        "first_date": price_df.apply(lambda s: s.dropna().index.min()),
        "last_date": price_df.apply(lambda s: s.dropna().index.max()),
        "n_obs": price_df.count(),
    })
    summary["years_span"] = (summary["last_date"] - summary["first_date"]).dt.days / 365.25

    print(f"--- {label} ---")
    display(summary)

    short = summary[summary["years_span"] < min_years]
    if not short.empty:
        print(f"WARNING: history shorter than {min_years} years for: {list(short.index)}")
    else:
        print(f"All assets meet the {min_years}-year minimum.")
    return summary

raw_summary = summarize_panel(raw_close, label="Raw close -- inspection", min_years=8)
adj_summary = summarize_panel(adj_close, label="Adjusted close -- inspection", min_years=8)

In [ ]:
ASSET_METADATA = {
    "AAPL":   {"name": "Apple Inc.",            "sector": "Information Technology",
               "exchange": "NASDAQ", "currency": "USD"},
    "JPM":    {"name": "JPMorgan Chase & Co.",  "sector": "Financials",
               "exchange": "NYSE",   "currency": "USD"},
    "XOM":    {"name": "ExxonMobil Corp.",      "sector": "Energy",
               "exchange": "NYSE",   "currency": "USD"},
    "JNJ":    {"name": "Johnson & Johnson",     "sector": "Health Care",
               "exchange": "NYSE",   "currency": "USD"},
    "7203.T": {"name": "Toyota Motor Corp.",    "sector": "Consumer Discretionary (Automobiles)",
               "exchange": "Tokyo Stock Exchange", "currency": "JPY"},
}

asset_summary = (
    pd.DataFrame(ASSET_METADATA).T
    .join(raw_summary[["first_date", "last_date", "n_obs", "years_span"]])
)
asset_summary.index.name = "ticker"
asset_summary["years_span"] = asset_summary["years_span"].round(3)

display(asset_summary)

n_obs_gap = asset_summary["n_obs"].max() - asset_summary["n_obs"].min()
if n_obs_gap > 0:
    shortest = asset_summary["n_obs"].idxmin()
    print(f"\n{shortest} has {n_obs_gap} fewer trading-day observations than the fullest "
          f"series over the same nominal date range -- direct evidence that its exchange "
          f"does not share a trading calendar with the US tickers.")

In [ ]:
def load_ohlcv_preview(tickers, start, end, cache_dir=DATA_DIR / "ohlcv_preview",
                       force_download=False):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / "ohlcv_long.csv"

    if not force_download and cache_path.exists():
        return pd.read_csv(cache_path, parse_dates=["date"])

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame")

        frames = []
        if isinstance(data.columns, pd.MultiIndex):
            for t in tickers:
                if t not in data.columns.get_level_values(0):
                    continue
                sub = data[t].reset_index().rename(columns={"Date": "date"})
                sub["ticker"] = t
                frames.append(sub)
        else:
            sub = data.reset_index().rename(columns={"Date": "date"})
            sub["ticker"] = tickers[0]
            frames.append(sub)

        long = pd.concat(frames, ignore_index=True)
        long = long[["date", "ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        long.to_csv(cache_path, index=False)
        return long

    except Exception as exc:
        print(f"OHLCV preview download failed ({exc!r}); this cell is display-only, "
              "skipping it gracefully -- it does not affect any other section.")
        return None

ohlcv_long = load_ohlcv_preview(TICKERS, PRICE_START, PRICE_END)

if ohlcv_long is not None:
    print("Sample of the raw downloaded data (first 3 rows per ticker):")
    display(ohlcv_long.groupby("ticker").head(3).set_index(["ticker", "date"]))

    print("\nDescriptive summary per ticker, full sample (Open/High/Low/Close/Adj Close/Volume):")
    ohlcv_stats = (
        ohlcv_long.groupby("ticker")[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        .agg(["mean", "min", "max"])
        .round(2)
    )
    display(ohlcv_stats)

### A1 discussion

**What arrived, and did it meet the requirement?** All five assets returned a full,
unbroken series. The four US tickers (AAPL, JPM, XOM, JNJ) span 2016-01-04 to
2024-12-31, with 2,264 trading days each (8.99 years). Toyota (7203.T) spans 2016-01-04
to 2024-12-30, with 2,222 observations (8.99 years). The raw and adjusted panels
returned identical date ranges and counts for every asset, so the adjustment itself did
not truncate anyone's history. Every asset clears the 8-year minimum. None triggered
`summarize_panel()`'s shortfall warning.

**Does Toyota's different calendar already show up here?** Yes. Toyota has 42 fewer
trading-day observations than the four US tickers, even though we requested the same
nominal date range for all five. That gap is direct evidence the Tokyo Stock Exchange
does not share a holiday calendar with the US market. Japanese New Year, Golden Week,
and other JP-only holidays fall on days the US market is open, and the reverse also
happens, though less often. This one number justifies the non-trading-day handling A2
Step 2 builds. It also shows why Toyota should be treated as a genuinely different
asset, a different currency and a different trading calendar, not just a fifth
large-cap with an unusual ticker.

## A2: Data and convention audit, the correction waterfall

Here's what we do in this section. We build a deliberately careless baseline, then
apply one correction at a time. Each correction builds on top of everything already
corrected before it. We recompute annualised return, annualised volatility, Sharpe
ratio, and maximum drawdown after every single step. We hold the portfolio construction
method fixed throughout: equal weight, no explicit rebalancing logic beyond what
`pct_change` implies. That way, the only thing changing row to row is the data or
convention correction under test, and you can see exactly what each one costs.

In [ ]:
def equal_weight_portfolio_returns(price_panel, return_type="simple"):
    asset_returns = price_panel.pct_change()
    portfolio_simple = asset_returns.mean(axis=1, skipna=True).dropna()
    if return_type == "simple":
        return portfolio_simple
    elif return_type == "log":
        return np.log1p(portfolio_simple)
    else:
        raise ValueError(return_type)

def compute_stats(returns, rf=0.0, periods_per_year=252, return_type="simple"):
    returns = returns.dropna()
    if isinstance(rf, pd.Series):
        rf = rf.reindex(returns.index).fillna(0.0)
    excess = returns - rf

    if return_type == "simple":
        wealth = (1 + returns).cumprod()
    elif return_type == "log":
        wealth = np.exp(returns.cumsum())
    else:
        raise ValueError(return_type)

    ann_return = returns.mean() * periods_per_year
    ann_vol = excess.std(ddof=1) * np.sqrt(periods_per_year)
    ann_excess_return = excess.mean() * periods_per_year
    sharpe = ann_excess_return / ann_vol if ann_vol > 0 else np.nan

    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1

    return {
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": drawdown.min(),
    }

waterfall_rows = {}

def record_step(label, portfolio_returns, **kwargs):
    stats = compute_stats(portfolio_returns, **kwargs)
    waterfall_rows[label] = stats
    print(label, "->", {k: round(v, 4) for k, v in stats.items()})
    return stats

In [ ]:
def naive_ffill(price_panel):
    return price_panel.ffill()

def defensible_fill(price_panel, max_gap=1):
    filled = price_panel.ffill(limit=max_gap)
    still_missing = filled.isna().sum()
    if still_missing.sum() > 0:
        print(f"Still missing after a {max_gap}-trading-day defensible fill "
              f"(left as NaN, not fabricated):")
        print(still_missing[still_missing > 0])
    else:
        print(f"No gaps longer than {max_gap} trading day(s) remained after the defensible fill.")
    return filled

In [ ]:
def flag_extreme_returns(asset_returns, z_thresh=6.0):
    z = (asset_returns - asset_returns.mean()) / asset_returns.std(ddof=1)
    mask = z.abs() > z_thresh
    records = []
    for ticker in asset_returns.columns:
        hits = asset_returns.loc[mask[ticker].fillna(False), ticker]
        for date, value in hits.items():
            records.append({"date": date, "ticker": ticker, "return": value,
                             "z_score": z.loc[date, ticker]})
    flagged = pd.DataFrame(records, columns=["date", "ticker", "return", "z_score"])
    return flagged.sort_values("date").reset_index(drop=True)

KNOWN_MARKET_EVENTS = [
    ("2018-02-02", "2018-02-09", "Volmageddon -- XIV unwind, VIX spike"),
    ("2018-12-01", "2018-12-26", "Q4 2018 growth-scare sell-off"),
    ("2020-02-20", "2020-04-07", "COVID-19 crash and initial rebound"),
    ("2020-09-01", "2020-09-08", "Sept 2020 tech pullback"),
    ("2020-11-09", "2020-11-09", "Pfizer/BioNTech vaccine efficacy announcement"),
    ("2022-01-01", "2022-10-31", "2022 rate-hike bear market"),
    ("2023-03-08", "2023-03-15", "SVB / regional bank stress"),
    ("2024-08-05", "2024-08-06", "BoJ rate hike / yen carry-trade unwind (see Lecture 6)"),
    ("2024-11-05", "2024-11-08", "2024 US presidential election result"),
]

def classify_flagged(flagged, events=KNOWN_MARKET_EVENTS):
    def event_for(date):
        for start, end, name in events:
            if pd.Timestamp(start) <= date <= pd.Timestamp(end):
                return name
        return None
    flagged = flagged.copy()
    flagged["known_event"] = flagged["date"].apply(event_for)
    return flagged

In [ ]:
def convert_jpy_to_usd(jpy_price_series, usdjpy_rate):
    return jpy_price_series / usdjpy_rate.reindex(jpy_price_series.index).ffill()

def align_risk_free(returns_index, rf_series, direction="backward"):
    left = pd.DataFrame(index=returns_index).rename_axis("date").reset_index()
    right = rf_series.sort_index().rename("rf").rename_axis("date").reset_index()
    aligned = pd.merge_asof(left, right, on="date", direction=direction).set_index("date")["rf"]
    return aligned

### Step 0: naive baseline

This is the deliberately careless starting point. We use raw, unadjusted closing
prices, blindly forward-filled with no gap limit, and compute a Sharpe ratio against a
risk-free rate of zero using the naive 252-trading-day annualisation assumption. Every
later step corrects exactly one thing on top of whatever came before it. Think of this
row as the "before" that everything else gets measured against.

In [ ]:
naive_prices = naive_ffill(raw_close)
r0 = equal_weight_portfolio_returns(naive_prices)
s0 = record_step("0. Naive baseline (raw prices, blind ffill, Sharpe vs 0, 252d, simple/arith.)", r0)

### Step 1: dividend- and split-adjusted prices

Raw closing prices don't account for dividends or stock splits. Both produce a price
change that isn't a real gain or loss to a continuously-holding investor. A 2-for-1
split halves the per-share price without changing anything about the position's value.
So we switch from raw close to the dividend/split-adjusted close, and hold everything
else exactly as it was in Step 0.

In [ ]:
adj_prices_naive_fill = naive_ffill(adj_close)
r1 = equal_weight_portfolio_returns(adj_prices_naive_fill)
s1 = record_step("1. + dividend/split-adjusted prices", r1)

### Step 2: defensible treatment of non-trading days

Toyota (7203.T, Tokyo Stock Exchange) and the four US tickers don't share a holiday
calendar. That means the raw joined panel has gaps on US-only and Japan-only holidays
alike. Blindly forward-filling every gap, as Steps 0 and 1 did, treats a several-day
data outage the same as a single foreign holiday. We think that's wrong, so instead we
forward-fill only single-day gaps, and report anything longer rather than fabricate
it.

In [ ]:
adj_prices_defensible = defensible_fill(adj_close, max_gap=1)
r2 = equal_weight_portfolio_returns(adj_prices_defensible)
s2 = record_step("2. + defensible non-trading-day treatment", r2)

### Step 3: investigate extreme returns

We flag any single-asset daily return more than 6 standard deviations from that asset's
own mean, then check each flagged date against known market-wide events in our sample.
If a flag falls inside a known event window, we treat it as a genuine market move and
leave it alone. Winsorising it would delete real information, which your brief
specifically warns against. If a flag does not correspond to a known event, we treat it
as a candidate data error. We correct those by replacing the price level with the most
recent valid price before it, a forward-fill, not the return, and only for the specific
date and ticker pairs identified.

We deliberately avoid a two-sided interpolation here, even though it would produce a
smoother-looking "corrected" price. Linear interpolation between the surrounding prices
uses the next valid observation as well as the prior one. That means the corrected
value on the flagged date would depend on information not yet available as of that
date. That's exactly the look-ahead bias this notebook hunts for elsewhere, for example
in Step 5's risk-free alignment bug below. Forward-filling from the last known good
price uses only information available at the time, at the cost of a slightly less
accurate point estimate of the "true" price on the corrected date. That's the same
trade-off Step 2 already accepted for ordinary non-trading-day gaps.

Worth telling you directly: our first pass at the known-events list only covered the
2018 sell-off and COVID. Run against the real data, it left five flagged observations
looking unexplained, and a naive pipeline would have interpolated over all five as if
they were data errors. Three of them weren't: the 9 November 2020 Pfizer vaccine
announcement, the 5 to 6 August 2024 Bank of Japan rate hike and yen carry-trade unwind
(the exact episode this subject's Lecture 6 covers under procyclicality), and the 6
November 2024 post-election financials rally. We caught this by checking every flag
against known events before trusting a "no correction needed" conclusion, and expanded
the event list below to the corrected version you now see.

In our own sample, this correction path never actually fires. Every flagged observation
below falls inside a known event window, so nothing gets corrected. The choice is
vacuous in effect here, but it's the principled, look-ahead-free choice for any future
run where it isn't.

One diagnostic worth watching for: a genuine one-sided crash doesn't fully reverse the
next day, while an isolated bad print typically shows up as a paired anomaly, an
extreme move immediately followed by a roughly offsetting one, as the series snaps back
to its true level. That pattern, if present, is evidence for "data error" over "real
move."

In [ ]:
asset_returns_2 = adj_prices_defensible.pct_change()
flagged = flag_extreme_returns(asset_returns_2, z_thresh=6.0)
flagged_classified = classify_flagged(flagged)
print(f"{len(flagged_classified)} extreme return(s) flagged (|z| > 6):")
display(flagged_classified)

genuinely_unexplained = flagged_classified[flagged_classified["known_event"].isna()]

if genuinely_unexplained.empty:
    print("\nEvery flagged extreme return falls inside a known market-wide event window "
          "(or none were flagged at all); none is treated as a data error. No price "
          "correction is applied at this step -- we demonstrate the check rather than "
          "assert its result. A vendor that served, say, a single-day zero-price glitch "
          "on an illiquid micro-cap would be exactly the case that WOULD move this row.")
    adj_prices_clean = adj_prices_defensible
else:
    print(f"\n{len(genuinely_unexplained)} flagged observation(s) do not correspond to a "
          "known market-wide event and are candidates for a genuine data error:")
    display(genuinely_unexplained)
    adj_prices_clean = adj_prices_defensible.copy()
    for _, row in genuinely_unexplained.iterrows():
        adj_prices_clean.loc[row["date"], row["ticker"]] = np.nan
    adj_prices_clean = adj_prices_clean.ffill()
    print("Replaced the flagged, unexplained observation(s) above with the most recent "
          "valid price (forward-fill, not a two-sided interpolation) -- using only "
          "information available as of the flagged date, to avoid introducing a "
          "look-ahead bias into the very correction meant to fix a data error.")

r3 = equal_weight_portfolio_returns(adj_prices_clean)
s3 = record_step("3. + investigated extreme returns", r3)

### Step 4: currency inconsistency

Four of our five assets are USD. Toyota (7203.T) is quoted in JPY. If we computed its
return directly from the JPY price and averaged it in with USD returns, we would
silently assume 1 JPY of price change is worth the same as 1 USD of price change, and
we would ignore the JPY/USD exchange-rate return entirely, which a USD-based investor
actually bears. So we convert Toyota's price series to USD using the USDJPY rate before
computing any returns.

One detail worth explaining: FX gaps in that rate series get forward-filled, not
treated the way an equity gap is. Currency trades continuously, so a missing daily
print just means the rate was unchanged since the last quote. That's a different
situation from an equity halt, where a gap could be hiding a real move.

In [ ]:
fx_raw, fx_adj, fx_dl_log = load_price_panel(["JPY=X"], PRICE_START, PRICE_END,
                                              cache_dir=DATA_DIR / "fx")
usdjpy = fx_raw["JPY=X"]
print("USDJPY download log:", fx_dl_log)

prices_usd = adj_prices_clean.copy()
prices_usd["7203.T"] = convert_jpy_to_usd(adj_prices_clean["7203.T"], usdjpy)

r4 = equal_weight_portfolio_returns(prices_usd)
s4 = record_step("4. + currency conversion (7203.T JPY -> USD)", r4)

### Step 5: introduce the risk-free rate (buggy, forward-looking alignment)

Every Sharpe ratio above was computed against a risk-free rate of exactly zero. We now
bring in the actual US 3-month T-bill yield (`^IRX`, quoted annualised, in percent) and
convert it to a per-period rate. T-bill yields aren't published on every trading day,
so aligning them onto our daily return index requires an as-of merge. That join
direction is a classic place for a look-ahead bug to hide. `direction="forward"` pairs
a return with a rate the market hadn't yet observed as of that date. We deliberately
introduce the rate this way first, so we can measure the bug's size in Step 6, rather
than fixing it silently and never showing you what it would have cost.

In [ ]:
rf_raw, rf_adj, rf_dl_log = load_price_panel(["^IRX"], PRICE_START, PRICE_END,
                                              cache_dir=DATA_DIR / "rf")
irx = rf_raw["^IRX"]
print("^IRX download log:", rf_dl_log)

rf_daily = (1 + irx / 100) ** (1 / 252) - 1
rf_forward_buggy = align_risk_free(r4.index, rf_daily, direction="forward")

s5 = record_step("5. + risk-free rate, buggy forward-looking alignment", r4, rf=rf_forward_buggy)

### Step 6: fix the look-ahead bug (backward-looking alignment)

`direction="backward"` uses the most recent rate that was actually known as of each
return date. No future information is used. We hold everything else identical to Step
5: the rate series itself, the portfolio, the annualisation factor. So any difference
in the statistics below is attributable to the join direction alone, isolating the
look-ahead effect from the separate "zero versus actual rate" effect we measure
explicitly below.

In [ ]:
rf_backward = align_risk_free(r4.index, rf_daily, direction="backward")
lookahead_gap = (rf_forward_buggy - rf_backward).abs()
print(f"Look-ahead alignment gap: mean={lookahead_gap.mean():.8f}, "
      f"max={lookahead_gap.max():.8f}, nonzero on "
      f"{int((lookahead_gap > 0).sum())} of {len(lookahead_gap)} dates")

s6 = record_step("6. + risk-free rate, corrected backward-looking alignment", r4, rf=rf_backward)

### Isolating one comparison: Sharpe against zero versus Sharpe against the actual cash rate

The waterfall above already contains this comparison implicitly. Step 4 used rf = 0;
Steps 5 and 6 use the actual T-bill rate. But your brief asks for it explicitly, so
here it is on its own. We hold the return series (`r4`), the annualisation factor
(252), and the alignment method (backward, correct) all fixed. Then we recompute the
same portfolio's Sharpe ratio twice, changing nothing but the risk-free assumption.

In [ ]:
sharpe_vs_zero = compute_stats(r4, rf=0.0, periods_per_year=252)["sharpe"]
sharpe_vs_actual = compute_stats(r4, rf=rf_backward, periods_per_year=252)["sharpe"]

rf_comparison = pd.DataFrame({
    "Risk-free assumption": ["Zero (rf = 0)", "Actual cash rate (^IRX, backward-aligned)"],
    "Sharpe ratio": [sharpe_vs_zero, sharpe_vs_actual],
}).set_index("Risk-free assumption")
rf_comparison["Change from zero-rf Sharpe"] = rf_comparison["Sharpe ratio"] - sharpe_vs_zero

display(rf_comparison.style.format({
    "Sharpe ratio": "{:.3f}", "Change from zero-rf Sharpe": "{:+.3f}",
}))

### Step 7: annualisation factor, 252 assumed versus actual trading days observed

In [ ]:
actual_ppy = len(r4) / ((r4.index[-1] - r4.index[0]).days / 365.25)
print(f"Actual periods/year in this panel: {actual_ppy:.2f} (naive assumption was 252)")

rf_daily_actual = (1 + irx / 100) ** (1 / actual_ppy) - 1
rf_backward_actual = align_risk_free(r4.index, rf_daily_actual, direction="backward")

s7 = record_step("7. + actual trading-day annualisation factor", r4,
                  rf=rf_backward_actual, periods_per_year=actual_ppy)

### Step 8: return definition, simple/arithmetic versus log/geometric

An arithmetic mean of simple returns, scaled to an annual figure, systematically
overstates the return an investor actually compounds to. That happens because
volatility drags the geometric growth rate below the arithmetic mean, Jensen's
inequality on the concave log-wealth function. Switching to log returns isn't a
separate "extra" correction stacked on top of computing a geometric mean. The
arithmetic mean of log returns already **is** the annualised geometric growth rate.
That's why this step recomputes everything with `return_type="log"` rather than adding
a fifth statistic.

Here's what this step does and doesn't change, and we want to be precise about it for
you. The underlying portfolio, and therefore its wealth path and maximum drawdown, is
identical to Step 7's. Only the summary statistic used to describe its annualised
return changes.

One thing worth telling you directly: an earlier draft of `equal_weight_portfolio_returns`
computed the log-return version by averaging each asset's own log return across the
cross-section. That's a subtly different quantity from this portfolio's own return on a
log scale, by Jensen's inequality. We confirmed the difference on a synthetic panel
with an injected single-asset shock: that mistake alone moved measured maximum drawdown
from -16.96% to -20.55%, purely as an averaging artefact, nothing to do with simple
versus log returns at all. We fixed it by taking `log1p` of the portfolio's own simple
return instead, so the wealth path stays identical to Step 7's by construction.

In [ ]:
log_returns = equal_weight_portfolio_returns(prices_usd, return_type="log")
s8 = record_step("8. + log returns (arithmetic mean = geometric growth rate)", log_returns,
                  rf=rf_backward_actual, periods_per_year=actual_ppy, return_type="log")

### The waterfall table

One row per correction, cumulative. This table is the deliverable for this section.

In [ ]:
waterfall = pd.DataFrame(waterfall_rows).T
waterfall.columns = ["Ann. return", "Ann. vol", "Sharpe", "Max drawdown"]
display(waterfall.style.format({
    "Ann. return": "{:.2%}", "Ann. vol": "{:.2%}",
    "Sharpe": "{:.3f}", "Max drawdown": "{:.2%}",
}))

### Do the corrections offset each other?

Compare the naive baseline to the fully corrected row: that's the net change. Then
compare that net change against the single largest step-to-step move for each
statistic. If the net change is noticeably smaller than the largest single step, some
corrections pushed the numbers in opposite directions. That would mean the
naive-versus-final comparison alone understates how wrong the naive figure actually was
at any single point in the pipeline.

In [ ]:
net_change = waterfall.iloc[-1] - waterfall.iloc[0]
step_changes = waterfall.diff().iloc[1:]

print("Net change, naive baseline -> fully corrected:")
print(net_change.round(4))

print("\nLargest single-step move per statistic:")
for col in waterfall.columns:
    step_name = step_changes[col].abs().idxmax()
    largest = step_changes.loc[step_name, col]
    ratio = abs(net_change[col]) / abs(largest) if largest != 0 else np.nan
    flag = " <-- net change is SMALLER than this single step: corrections partly offset" \
        if abs(net_change[col]) < abs(largest) - 1e-12 else ""
    print(f"  {col}: largest move at '{step_name}' ({largest:+.4f}); "
          f"net/largest ratio = {ratio:.2f}{flag}")

### A2 discussion: which correction mattered most?

**Which single correction moved the numbers most?** The dividend/split adjustment
(Step 1) moved three of the four statistics more than any other step. It took the
Sharpe ratio from 0.855 to 1.018 (+0.163), annualised return from 14.19% to 16.90%
(+2.71pp), and improved maximum drawdown from -35.08% to -34.78% (+0.30pp). Only
annualised volatility moved more under a different step: the trading-day annualisation
factor (Step 7, +0.27pp).

**Was the naive Sharpe ratio a modest overstatement, or a fundamentally different
answer?** Neither, cleanly, and that is itself the finding. The net change from naive
(0.855) to fully corrected (0.800) looks small: -0.055, about a 6% relative decrease.
Read on its own, that suggests the naive figure was basically fine. It was not. The
dividend adjustment alone raised the Sharpe ratio by 16 percentage points (0.855 to
1.018), meaning the naive baseline actually understated it relative to a
partially-corrected figure. Three later corrections then pulled it back down past the
naive starting point: currency conversion (-0.035), the risk-free rate (-0.115, mostly
from moving off a zero-rate assumption rather than from the look-ahead bug), and the
switch to log/geometric returns (-0.085). The small net change is a coincidence of
these swings roughly cancelling out. It is not evidence the naive figure was
defensible.

**Did corrections offset each other?** Yes, substantially, exactly as your brief
warns. Sharpe's net move (-0.055) is only about a third of its largest single step
(+0.163), strong evidence of offsetting. Annualised return's net move (+1.18pp) is 44%
of its largest single step (+2.71pp). Maximum drawdown's net move (+0.22pp) is 73% of
its largest single step (+0.30pp). Only annualised volatility moved in a fairly
consistent direction throughout, with a net/largest ratio of 85%, because nearly every
correction from Step 1 onward pushed volatility up rather than in mixed directions.

The mechanism is concrete, and we want to walk you through it. Dividend/split
adjustment removes phantom return spikes that inflate both the mean and, through those
same spikes, the volatility of the naive series. That mechanically raises Sharpe.
Currency conversion, the non-zero risk-free rate, and the geometric-mean correction all
remove a different kind of overstatement, one the naive baseline never had a mechanism
to create in the first place. They are independent sources of downward correction, not
linked to the dividend fix by any real economic relationship. The near-cancellation we
see is a coincidence of this specific universe and this 2016-2025 window. A different
asset mix, or a period with a larger split/dividend history relative to FX and rate
effects, could easily push every correction the same direction. In that case the
naive-vs-final gap would be far larger than any individual step. This is exactly your
brief's own warning in action: a bare before-and-after comparison is untrustworthy, and
this table proves it, not just states it.

### Survivorship bias

**What it is.** Every asset in our universe, Apple, JPMorgan, ExxonMobil, Johnson &
Johnson, Toyota, still exists today and is a well-known, currently-thriving large-cap.
We selected them with the benefit of nine years of hindsight. A retail data vendor like
Yahoo Finance only serves prices for tickers that are still listed, or were listed
under the same symbol until a gracefully-handled event like a merger. It does not serve
a point-in-time-correct universe of everything that was investable on 1 January 2016,
including firms that have since been delisted, gone bankrupt, or been acquired at a
distressed price. We cannot fix this with the data available to us: `yfinance` has no
route to a name that no longer trades.

**Direction of the bias.** It pushes return and Sharpe ratio up, and understates
volatility and drawdown. A portfolio that can only ever hold survivors never
experiences one of its constituents going to zero. So its realised return, Sharpe
ratio, and worst drawdown all look more flattering than what an investor holding the
actual, point-in-time investable universe in 2016 would have experienced.

**Rough magnitude.** Lecture 2 cites a calibrated delisting simulation on a broad,
systematically-sampled S&P 500 backtest, putting survivorship bias at roughly +2.7%
p.a. of phantom return. Our case is arguably worse, not better. We did not sample
systematically at all. We hand-picked five globally recognised, multi-decade survivors
specifically because they were recognisable. That is a stronger hindsight-driven filter
than simply "was still in the S&P 500 index at the download date." We would expect our
true survivorship effect, if we could measure it, to sit at or above that +2.7% p.a.
anchor rather than below it. We have no way to quantify our own figure, though, without
a point-in-time delisted-security database, and that kind of database is not available
through a free retail vendor.

## A3: Baseline risk report

We use the corrected data throughout. `prices_usd` from A2 is our clean price panel
from here on: adjusted for corporate actions, defensibly filled, checked for data
errors, and currency-converted. `r4`, the equal-weight portfolio return computed from
it in A2 Step 4, is our portfolio return series. We add a per-asset return panel
alongside it, so you can see each asset next to the portfolio.

In [ ]:
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns

asset_returns_final = prices_usd.pct_change()
portfolio_returns = r4

returns_for_a3 = {t: asset_returns_final[t] for t in TICKERS}
returns_for_a3["Portfolio (1/N)"] = portfolio_returns

print("Series available for A3:", list(returns_for_a3.keys()))

### Return characteristics: moments and normality

Here we report mean, volatility, skewness, and excess kurtosis for each asset and the
portfolio, plus a Jarque-Bera test of normality. That test is built directly from
sample skewness and kurtosis, so its result should read consistently with the moments
sitting right next to it.

In [ ]:
def compute_return_moments(returns_dict, periods_per_year=252):
    rows = {}
    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()
        jb_stat, jb_p = stats.jarque_bera(r)
        rows[name] = {
            "mean_daily": r.mean(),
            "vol_daily": r.std(ddof=1),
            "ann_return": r.mean() * periods_per_year,
            "ann_vol": r.std(ddof=1) * np.sqrt(periods_per_year),
            "skewness": stats.skew(r),
            "excess_kurtosis": stats.kurtosis(r),
            "jarque_bera_stat": jb_stat,
            "jarque_bera_pvalue": jb_p,
        }
    return pd.DataFrame(rows).T

moments_table = compute_return_moments(returns_for_a3, periods_per_year=actual_ppy)
display(moments_table.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))

n_reject_normality = (moments_table["jarque_bera_pvalue"] < 0.05).sum()
print(f"\n{n_reject_normality} of {len(moments_table)} series reject normality at the 5% level.")

**What the higher moments imply.** All six series, every asset and the portfolio,
reject normality decisively (Jarque-Bera p < 0.0001 throughout). Excess kurtosis ranges
from 5.9 (AAPL, 7203.T) to 14.9 (JPM), against 0 for a true normal distribution. These
are not marginal deviations. They are an order of magnitude beyond what a Gaussian
model allows. JPM's kurtosis stands out: it had the largest flagged extreme moves in A2
(the Nov 2020 vaccine rally and the Nov 2024 post-election rally both hit JPM
specifically). That same fat-tailedness shows up again below. JPM also has the largest
historical-vs-parametric ES gap at 99%. Two independent measurements agree on the same
underlying property, and that's a useful consistency check for you to see.

Skewness is more mixed, and genuinely interesting. Four of the five individual assets
show slightly positive skew (AAPL +0.001, JPM +0.421, XOM +0.069, 7203.T +0.229). JNJ
(-0.180) and, notably, the portfolio (-0.299) show negative skew. A diversified
portfolio having more negative skew than most of its own constituents looks
counterintuitive at first, but the correlation analysis below explains it. Individual
stocks can show positive skew from idiosyncratic upside jumps, like earnings surprises
or single-name rallies. When a systemic shock hits, though, our five assets stop
behaving idiosyncratically and crash together. The portfolio inherits a left-tail event
risk that is largely invisible when you look at each asset's own return distribution in
isolation. This is exactly why any risk measure that assumes a Gaussian, symmetric,
thin-tailed distribution, including the parametric VaR/ES computed next, will
understate genuine portfolio-level tail risk here. The danger isn't in each asset's own
distribution. It's in what happens to their joint distribution during a crisis.

### Value at Risk and Expected Shortfall

We compute two methods, at 95% and 99% confidence, so you can see where they agree and
where they don't:

- **Historical (empirical):** the actual sample quantile of the loss distribution, and
  the average loss beyond it. This makes no distributional assumption, but it's only as
  good as the historical sample's coverage of tail events.
- **Parametric (Gaussian):** assumes returns are normally distributed and computes
  VaR/ES from the sample mean and standard deviation in closed form. It's cheap and
  smooth, but it rests on exactly the assumption the moments table above is testing.

Both are reported as positive loss numbers. VaR95 = 5.2% means a 5.2% loss is exceeded
5% of the time.

In [ ]:
def historical_var_es(returns, alpha=0.95):
    r = pd.Series(returns).dropna()
    q = r.quantile(1 - alpha)
    var = -q
    es = -r[r <= q].mean()
    return var, es

def parametric_var_es(returns, alpha=0.95):
    r = pd.Series(returns).dropna()
    mu, sigma = r.mean(), r.std(ddof=1)
    q = norm.ppf(1 - alpha)
    var = -(mu + sigma * q)
    es = -mu + sigma * norm.pdf(q) / (1 - alpha)
    return var, es

def var_es_table(returns_dict, confidence_levels=(0.95, 0.99)):
    rows = []
    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()
        for alpha in confidence_levels:
            hv, he = historical_var_es(r, alpha)
            pv, pe = parametric_var_es(r, alpha)
            rows.append({"asset": name, "confidence": f"{alpha:.0%}",
                         "method": "historical", "VaR": hv, "ES": he})
            rows.append({"asset": name, "confidence": f"{alpha:.0%}",
                         "method": "parametric", "VaR": pv, "ES": pe})
    return pd.DataFrame(rows)

var_es = var_es_table(returns_for_a3, confidence_levels=(0.95, 0.99))
var_es_wide = var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                  values=["VaR", "ES"])
display(var_es_wide.style.format("{:.2%}"))

disagreement = (var_es.pivot_table(index=["asset", "confidence"], columns="method", values="ES")
                .assign(gap=lambda d: d["historical"] - d["parametric"]))
print("\nHistorical minus parametric ES (positive = parametric understates the tail):")
display(disagreement.style.format("{:.2%}"))

**Which would we report to a risk committee?** The historical-minus-parametric gap is
positive for every asset and the portfolio, at both confidence levels. The Gaussian
assumption understates tail risk uniformly across this universe, not just on average.
The gap roughly triples to quadruples moving from 95% to 99% confidence: for the
portfolio, 0.36pp at 95% versus 1.57pp at 99%; for JPM, 0.41pp versus 2.03pp. This
matches exactly what the moments table predicts. Fat tails matter most in the deepest
part of the tail, which is precisely where the 99% figure lives and where the Gaussian
approximation is weakest.

We would report the historical ES, not the parametric figure, as the headline number to
a risk committee. Two reasons reinforce each other here. First, the Jarque-Bera test
has already rejected the assumption the parametric method depends on, for every series
in the portfolio, so there is no basis for trusting it. Second, understating a 1-in-100
loss is the more expensive mistake for a risk committee to make than overstating one. It
directly under-capitalises for the exact scenario a risk limit exists to catch: JPM's
99% ES is 6.55% historical versus 4.52% parametric, roughly 45% higher. The parametric
number is still worth showing alongside it, clearly labelled as a normality-assuming
sanity check. Its gap from the historical figure is itself informative about how
fat-tailed the true distribution is. But it should never be the number a limit is set
against.

### Drawdown analysis

One definition worth stating up front, since it isn't obvious from the chart alone: the
peak we report is the last date at or before the trough where wealth equalled its own
running maximum. That's the date the subsequent decline actually started, not just any
earlier high point.

In [ ]:
def drawdown_series(returns):
    wealth = (1 + pd.Series(returns).dropna()).cumprod()
    running_max = wealth.cummax()
    dd = wealth / running_max - 1
    trough_date = dd.idxmin()
    max_dd = dd.loc[trough_date]
    peak_date = wealth.loc[:trough_date].idxmax()
    return dd, {"max_drawdown": max_dd, "peak_date": peak_date, "trough_date": trough_date}

portfolio_dd, portfolio_dd_info = drawdown_series(portfolio_returns)
print("Portfolio maximum drawdown:")
for k, v in portfolio_dd_info.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(portfolio_dd.index, portfolio_dd.values * 100, 0, color="#C0392B", alpha=0.5)
ax.plot(portfolio_dd.index, portfolio_dd.values * 100, color="#C0392B", linewidth=0.8)
ax.axvline(portfolio_dd_info["peak_date"], color="black", linestyle="--", linewidth=1,
           label=f"peak ({portfolio_dd_info['peak_date'].date()})")
ax.axvline(portfolio_dd_info["trough_date"], color="black", linestyle=":", linewidth=1,
           label=f"trough ({portfolio_dd_info['trough_date'].date()})")
ax.set_title("Portfolio drawdown, 1/N equal-weight, corrected data")
ax.set_xlabel("date")
ax.set_ylabel("drawdown (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Identifying the drawdown.** Peak 20 January 2020, trough 23 March 2020, maximum
drawdown -34.86%. This is unambiguously the COVID-19 crash. It falls squarely inside the
`KNOWN_MARKET_EVENTS` window from A2 ("2020-02-20" to "2020-04-07"), and 23 March 2020
is the well-documented historical bottom of the broad US equity market. That gives us a
useful external check on the drawdown code itself, independent of anything internal to
this notebook. The chart also shows two other, shallower episodes that match our event
list: a roughly -20% drawdown around Q4 2018 (the growth-scare sell-off) and a roughly
-18% drawdown through 2022 (the rate-hike bear market). Neither approaches COVID's
depth, but both land exactly where the literature and our own A2 event list say they
should.

### Correlation structure

We show the full-sample correlation matrix, then a rolling view. Diversification is a
claim about the whole sample, but the number that actually matters to a risk manager is
whether it holds up exactly when it's needed, during the worst drawdown.

In [ ]:
corr_matrix = asset_returns_final[TICKERS].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, ax=ax, cbar_kws={"label": "correlation"})
ax.set_title("Full-sample correlation matrix (corrected USD returns)")
plt.tight_layout()
plt.show()

display(corr_matrix.style.format("{:.3f}"))

In [ ]:
def rolling_avg_pairwise_corr(returns_df, window=60):
    cols = list(returns_df.columns)
    pairs = [(a, b) for i, a in enumerate(cols) for b in cols[i + 1:]]
    rolling_corrs = pd.DataFrame({
        f"{a}-{b}": returns_df[a].rolling(window).corr(returns_df[b])
        for a, b in pairs
    })
    avg_corr = rolling_corrs.mean(axis=1)
    return avg_corr, rolling_corrs

ROLLING_CORR_WINDOW = 60
avg_corr, pairwise_corrs = rolling_avg_pairwise_corr(asset_returns_final[TICKERS],
                                                      window=ROLLING_CORR_WINDOW)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                          gridspec_kw={"height_ratios": [1, 1.3]})

axes[0].fill_between(portfolio_dd.index, portfolio_dd.values * 100, 0,
                      color="#C0392B", alpha=0.4)
axes[0].set_ylabel("drawdown (%)")
axes[0].set_title("Portfolio drawdown vs. rolling average pairwise correlation")

axes[1].plot(avg_corr.index, avg_corr.values, color="#123F69", linewidth=1.2)
axes[1].axhline(avg_corr.mean(), color="grey", linestyle=":", linewidth=1,
                 label=f"full-sample average ({avg_corr.mean():.2f})")
axes[1].set_ylabel(f"{ROLLING_CORR_WINDOW}d avg pairwise correlation")
axes[1].set_xlabel("date")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.axvspan(portfolio_dd_info["peak_date"], portfolio_dd_info["trough_date"],
               color="black", alpha=0.08, label="max-drawdown window")

plt.tight_layout()
plt.show()

In [ ]:
dd_window_corr = avg_corr.loc[portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]].mean()
full_sample_corr = avg_corr.mean()
print(f"Average pairwise correlation during the max-drawdown window "
      f"({portfolio_dd_info['peak_date'].date()} to {portfolio_dd_info['trough_date'].date()}): "
      f"{dd_window_corr:.3f}")
print(f"Average pairwise correlation over the full sample: {full_sample_corr:.3f}")
print(f"Difference: {dd_window_corr - full_sample_corr:+.3f} "
      f"({'higher' if dd_window_corr > full_sample_corr else 'lower'} during the drawdown)")

**Did diversification fail when it mattered?** Unambiguously yes. The full-sample
average pairwise correlation is 0.21. During the COVID drawdown window it climbs to
roughly 0.70 at its peak, more than three times the baseline, before decaying back into
the 0.1 to 0.3 range that characterises the rest of the sample. This is direct,
sample-specific confirmation of the "correlations rise in crises" stylised fact from
Lecture 6, not just a citation of it. Our five-asset, sector- and currency-diversified
universe offered substantially less protection during its single worst episode than the
full-sample correlation matrix would suggest on its own.

This also explains the portfolio's negative skewness noted above. The joint crash that
drives the left tail is exactly this same correlation spike, invisible in any one
asset's own return distribution, but fully visible here. The rolling series also shows
correlation was already trending upward through late 2019, before the drawdown itself
began, and elevated again around 2022 to 2023. That suggests this is a recurring
pattern in our sample, not a one-off coincidence of COVID specifically. Toyota's low
full-sample correlation with the US names (0.06 to 0.16, the lowest pairs in the
matrix) is real diversification value in normal times. But the rolling chart is the
honest caveat, and we want to be direct with you about it: that diversification does
not fully survive contact with a genuinely global, systemic shock, where even a
JPY-denominated, Tokyo-listed automaker moved with the rest of the panel.

## A4: The equal-weight benchmark

**Rebalancing rule: monthly.** On the first trading day of every calendar month, we
reset weights to exactly 1/N. Between rebalance dates, we let weights drift with asset
prices, buy-and-hold. Monthly is a middle ground. Annual lets drift accumulate for a
long time, and A3's evidence shows correlation and volatility regimes can shift well
within a year. Daily rebalancing, which is what A2 and A3 used implicitly throughout,
overstates what a real investor would do, since it assumes costless, frictionless
trading every single day. Monthly is also the more common real-world convention for a
passive benchmark.

One thing worth flagging directly: this makes A2/A3's portfolio (`r4`) and A4's
benchmark genuinely different series, not the same thing under a new name. A2/A3 used
the daily-implicit version throughout because A2's question was about data corrections,
not rebalancing policy. From here on, `benchmark_returns` (monthly-rebalanced) is what
we compare Part B against, per your instruction.

In [ ]:
def rebalanced_portfolio_returns(price_panel, rebalance_freq="M", weights=None):
    returns = price_panel.pct_change().dropna(how="all")
    tickers = list(returns.columns)
    n = len(tickers)
    if weights is None:
        weights = {t: 1.0 / n for t in tickers}
    target = np.array([weights[t] for t in tickers])

    period_arr = np.asarray(returns.index.to_period(rebalance_freq))
    is_rebalance_day = np.empty(len(period_arr), dtype=bool)
    is_rebalance_day[0] = True
    is_rebalance_day[1:] = period_arr[1:] != period_arr[:-1]

    R = returns[tickers].values
    w = target.copy()
    port_returns = np.empty(len(returns))
    turnover = np.empty(len(returns))

    for t in range(len(returns)):
        if is_rebalance_day[t]:
            turnover[t] = np.abs(target - w).sum() / 2.0
            w = target.copy()
        else:
            turnover[t] = 0.0
        r_t = np.nan_to_num(R[t], nan=0.0)
        port_ret = float(np.dot(w, r_t))
        port_returns[t] = port_ret
        w = w * (1 + r_t) / (1 + port_ret)

    return (pd.Series(port_returns, index=returns.index, name="benchmark"),
            pd.Series(turnover, index=returns.index, name="turnover"))

benchmark_returns, benchmark_turnover = rebalanced_portfolio_returns(prices_usd, rebalance_freq="M")
n_rebalances = (benchmark_turnover > 0).sum()
print(f"Monthly-rebalanced benchmark built: {len(benchmark_returns)} daily observations, "
      f"{n_rebalances} rebalance events.")
print(f"Average one-way turnover per rebalance event: {benchmark_turnover[benchmark_turnover > 0].mean():.2%}")

### Benchmark performance and risk profile

We use the same statistics as A3: return moments and normality, VaR/ES by two methods,
and the drawdown series, all applied to the monthly-rebalanced benchmark. We also place
the daily-implicit portfolio (`r4`) from A2/A3 alongside it, so you can see the effect
of the rebalancing policy itself directly, rather than take our word for it.

In [ ]:
benchmark_moments = compute_return_moments(
    {"Benchmark (1/N, monthly)": benchmark_returns, "Daily-implicit (A2/A3)": portfolio_returns},
    periods_per_year=actual_ppy,
)
display(benchmark_moments.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))

In [ ]:
benchmark_var_es = var_es_table(
    {"Benchmark (1/N, monthly)": benchmark_returns, "Daily-implicit (A2/A3)": portfolio_returns},
    confidence_levels=(0.95, 0.99),
)
benchmark_var_es_wide = benchmark_var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                                      values=["VaR", "ES"])
display(benchmark_var_es_wide.style.format("{:.2%}"))

In [ ]:
benchmark_dd, benchmark_dd_info = drawdown_series(benchmark_returns)
print("Benchmark (1/N, monthly) maximum drawdown:")
for k, v in benchmark_dd_info.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(portfolio_dd.index, portfolio_dd.values * 100, color="#9AA5B1", linewidth=0.9,
        label="daily-implicit (A2/A3)")
ax.plot(benchmark_dd.index, benchmark_dd.values * 100, color="#123F69", linewidth=1.1,
        label="benchmark (1/N, monthly-rebalanced)")
ax.set_title("Drawdown: monthly-rebalanced benchmark vs. daily-implicit portfolio")
ax.set_xlabel("date")
ax.set_ylabel("drawdown (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Rebalancing frequency barely matters here.** The monthly benchmark and the
daily-implicit portfolio are close to indistinguishable on every statistic: annualised
return 16.40% vs 16.79% (-0.39pp), annualised vol 16.68% vs 16.80% (-0.12pp), and every
VaR/ES figure at both confidence levels agrees to within 0.01 to 0.02pp. The drawdown is
the most striking case. Peak (2020-01-20) and trough (2020-03-23) land on the exact same
day for both series, with maximum drawdown differing by only 0.01pp (-34.87% vs
-34.86%). The chart shows the two lines essentially overlapping for the entire nine-year
sample.

This makes sense given how the two are constructed. Monthly rebalancing only lets
weights drift for at most about 21 trading days before resetting. That is not long
enough for our five moderately-correlated assets to diverge far from 1/N before being
pulled back. So the reported risk and return statistics are nearly
rebalancing-frequency-invariant over this horizon. What genuinely differs is turnover,
and therefore cost. The benchmark generated 107 rebalance events at 1.97% average
one-way turnover each, roughly 23% of one-way turnover per year (107 times 1.97%,
divided by about 9.16 years). The daily-implicit version, by contrast, has the
effective turnover of resetting to 1/N every single trading day: an order of magnitude
more trading for statistically indistinguishable results. That is the concrete case for
monthly over daily as a benchmark policy. It is the cheaper way to deliver essentially
the same risk-adjusted outcome, which is exactly the property you want in something
Part B has to beat.

### Frictions ignored

You don't require us to model these, but you do require us to know they're missing,
and to have a sense of their size rather than just naming them.

- **Transaction costs.** The benchmark trades on every rebalance date, at the average
  one-way turnover per event reported by the "benchmark built" cell above, roughly a
  few percent per month, driven by how far our five assets drift apart between
  rebalances. Even a conservative 5 to 10 bp round-trip cost on large-cap, liquid names
  adds up to a small but nonzero annual drag. It would be far larger for a less liquid
  universe or a more frequent rebalancing rule.
- **Bid-ask spread.** Every trade crosses the spread, which is not the same as a
  commission and is not visible in any exchange-reported closing price. It's a real
  cost this notebook has no way to measure from daily OHLCV data alone.
- **Cash drag from imperfect rebalancing.** A real account can't buy fractional shares
  in the exact proportions 1/N implies, and can't rebalance at the exact instant the
  code specifies, since the next tradable price isn't the theoretical rebalance price.
  Both push the achievable return slightly below what this simulation reports.

None of these favour the benchmark disproportionately. If anything, an actively
rebalanced or optimised Part B strategy typically trades more than 1/N, so these same
frictions bite it harder. That asymmetry is itself relevant context for evaluating Part
B against this benchmark, not just a disclaimer.

### Adopting the benchmark for Part B

`benchmark_returns` (1/N, monthly-rebalanced) is the reference series for every Part B
result that can sensibly be compared against it, as your brief requires. We build one
small, reusable comparison function here, rather than re-deriving it per extension.

In [ ]:
def benchmark_comparison_table(strategy_returns_dict, benchmark=benchmark_returns,
                               periods_per_year=None, confidence_levels=(0.95, 0.99)):
    ppy = periods_per_year if periods_per_year is not None else actual_ppy
    series = {"Benchmark (1/N, monthly)": benchmark, **strategy_returns_dict}

    moments = compute_return_moments(series, periods_per_year=ppy)
    var_es = var_es_table(series, confidence_levels=confidence_levels)
    var_es_wide = var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                     values=["VaR", "ES"])

    dd_rows = {}
    for name, r in series.items():
        _, info = drawdown_series(r)
        dd_rows[name] = info
    drawdowns = pd.DataFrame(dd_rows).T

    return moments, var_es_wide, drawdowns

_moments_check, _var_es_check, _dd_check = benchmark_comparison_table({})
display(_moments_check.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))

## Extension 1: Portfolio optimisation (Category B, option B1)

### Methodological choices

**Short selling: not permitted, long-only.** Two reasons. First, Jagannathan and Ma
(2003) show a no-short constraint is mathematically equivalent to shrinking the
covariance matrix. It's not merely a practical restriction, it's itself a defence
against estimation error, and A3 already established that this universe has heavy fat
tails and a correlation structure that is anything but stable through time. Second, a
long-only mandate is the realistic default for the kind of investor this benchmark
exercise represents.

**Expected returns: the simple historical sample mean, annualised.** This is
deliberately the weakest possible choice, and we want to be upfront about that rather
than hide it. Chopra and Ziemba (1993) find errors in mu are roughly an order of
magnitude more damaging to portfolio choice than errors in the covariance matrix, and
Lecture 2 already established that returns are close to unforecastable at this horizon.
We use it anyway, precisely so the in-sample/out-of-sample comparison below can show you
what that weak link actually costs, rather than assume a better forecast we don't have.

**Covariance estimator: Ledoit-Wolf shrinkage**, not the raw sample covariance. Even
though T is much larger than N here, roughly 2,000+ daily observations against 5
assets, unlike the high-dimensional factor-zoo setting shrinkage is usually motivated
by, shrinkage is close to costless and never hurts conditioning. It's the honest
baseline any more elaborate covariance method should be measured against, per Lecture
3.

**Risk parity's objective**, for completeness: we solve the Spinu (2013) convex
reformulation from Lecture 6, minimising `0.5 w'Sigma w - sum(b_i log w_i)`, which
recovers equal risk contributions once you rescale the weights to sum to 1. We verify
this against a synthetic portfolio with known unequal volatilities, checking realised
risk contributions directly rather than trusting solver convergence alone, since Part C
explains exactly why that check matters.

In [ ]:
from sklearn.covariance import LedoitWolf

def estimate_covariance(returns_df, method="ledoit_wolf", periods_per_year=252):
    clean = returns_df.dropna()
    X = clean.values
    if method == "sample":
        cov = np.cov(X, rowvar=False, ddof=1)
    elif method == "ledoit_wolf":
        cov = LedoitWolf().fit(X).covariance_
    else:
        raise ValueError(method)
    return pd.DataFrame(cov, index=clean.columns, columns=clean.columns) * periods_per_year

In [ ]:
from scipy.optimize import minimize

def optimize_portfolio(mu, cov, objective="min_variance", rf=0.0, target_return=None,
                       long_only=True):
    tickers = mu.index if isinstance(mu, pd.Series) else None
    mu_v, cov_v = np.asarray(mu), np.asarray(cov)
    n = len(mu_v)
    x0 = np.full(n, 1.0 / n)

    if objective == "risk_parity":
        b = np.full(n, 1.0 / n)
        bounds = [(1e-8, None)] * n
        def rp_obj(w):
            return 0.5 * w @ cov_v @ w - np.sum(b * np.log(w))
        res = minimize(rp_obj, x0, method="SLSQP", bounds=bounds,
                       options={"maxiter": 1000, "ftol": 1e-14})
        w = res.x / res.x.sum()
        return (pd.Series(w, index=tickers) if tickers is not None else w), res

    bounds = [(0.0, 1.0)] * n if long_only else [(-1.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]
    if target_return is not None:
        constraints.append({"type": "eq", "fun": lambda w: w @ mu_v - target_return})

    if objective == "min_variance":
        obj = lambda w: w @ cov_v @ w
    elif objective == "max_sharpe":
        def obj(w):
            vol = np.sqrt(w @ cov_v @ w)
            return -(w @ mu_v - rf) / vol if vol > 1e-12 else 1e6
    else:
        raise ValueError(objective)

    res = minimize(obj, x0, method="SLSQP", bounds=bounds, constraints=constraints,
                   options={"maxiter": 1000, "ftol": 1e-12})
    if not res.success:
        print(f"WARNING: optimizer did not converge cleanly for {objective}: {res.message}")
    w = res.x
    return (pd.Series(w, index=tickers) if tickers is not None else w), res

def risk_contributions(w, cov):
    w_v, cov_v = np.asarray(w), np.asarray(cov)
    port_vol = np.sqrt(w_v @ cov_v @ w_v)
    marginal = (cov_v @ w_v) / port_vol
    return pd.Series(w_v * marginal, index=w.index if isinstance(w, pd.Series) else None)

In [ ]:
mu_full = asset_returns_final[TICKERS].mean() * actual_ppy
cov_full = estimate_covariance(asset_returns_final[TICKERS], "ledoit_wolf", periods_per_year=actual_ppy)

w_minvar, res_minvar = optimize_portfolio(mu_full, cov_full, objective="min_variance")
w_maxsharpe, res_maxsharpe = optimize_portfolio(mu_full, cov_full, objective="max_sharpe")
w_riskparity, res_riskparity = optimize_portfolio(mu_full, cov_full, objective="risk_parity")
w_equal = pd.Series(1.0 / len(TICKERS), index=TICKERS)

for name, res in [("min_variance", res_minvar), ("max_sharpe", res_maxsharpe),
                  ("risk_parity", res_riskparity)]:
    print(f"{name}: converged = {res.success}")

weights_table = pd.DataFrame({
    "Min variance": w_minvar, "Max Sharpe": w_maxsharpe,
    "Risk parity": w_riskparity, "1/N": w_equal,
})
display(weights_table.style.format("{:.2%}"))

rc_table = pd.DataFrame({
    "Min variance": risk_contributions(w_minvar, cov_full) / np.sqrt(w_minvar.values @ cov_full.values @ w_minvar.values),
    "Max Sharpe": risk_contributions(w_maxsharpe, cov_full) / np.sqrt(w_maxsharpe.values @ cov_full.values @ w_maxsharpe.values),
    "Risk parity": risk_contributions(w_riskparity, cov_full) / np.sqrt(w_riskparity.values @ cov_full.values @ w_riskparity.values),
    "1/N": risk_contributions(w_equal, cov_full) / np.sqrt(w_equal.values @ cov_full.values @ w_equal.values),
})
print("\nRisk contribution shares (should be exactly equal within 'Risk parity' column):")
display(rc_table.style.format("{:.2%}"))

### Efficient frontier

In [ ]:
def efficient_frontier(mu, cov, n_points=40, long_only=True):
    mu_v, cov_v = np.asarray(mu), np.asarray(cov)
    targets = np.linspace(mu_v.min(), mu_v.max(), n_points)
    rows = []
    for tr in targets:
        w, res = optimize_portfolio(mu, cov, objective="min_variance",
                                    target_return=tr, long_only=long_only)
        if res.success:
            w_v = np.asarray(w)
            rows.append({"target_return": tr, "return": w_v @ mu_v,
                        "vol": np.sqrt(w_v @ cov_v @ w_v)})
    return pd.DataFrame(rows)

frontier = efficient_frontier(mu_full, cov_full, n_points=40)

gmv_return = frontier.loc[frontier["vol"].idxmin(), "return"]
frontier_efficient = frontier[frontier["return"] >= gmv_return - 1e-9]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frontier["vol"] * 100, frontier["return"] * 100, color="#9AA5B1",
        linewidth=1.2, linestyle="--", label="frontier (full, incl. inefficient half)")
ax.plot(frontier_efficient["vol"] * 100, frontier_efficient["return"] * 100,
        color="#123F69", linewidth=2, label="efficient frontier")

markers = {
    "Min variance": (w_minvar, "o", "#1B7F79"),
    "Max Sharpe": (w_maxsharpe, "^", "#C0392B"),
    "Risk parity": (w_riskparity, "s", "#8E6C1F"),
    "1/N benchmark": (w_equal, "D", "#E8871A"),
}
for label, (w, marker, color) in markers.items():
    w_v = w.values
    vol_pt = np.sqrt(w_v @ cov_full.values @ w_v) * 100
    ret_pt = (w_v @ mu_full.values) * 100
    ax.scatter([vol_pt], [ret_pt], marker=marker, s=110, color=color,
              edgecolor="black", linewidth=0.8, zorder=5, label=label)

ax.set_xlabel("annualised volatility (%)")
ax.set_ylabel("annualised return (%)")
ax.set_title("Efficient frontier (long-only, Ledoit-Wolf covariance, full sample)")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

**Concentration and risk allocation.** Max Sharpe is the clearest case of the
estimation-error sensitivity Lecture 3 warns about. It puts 53.39% of capital in AAPL,
the individual asset with the highest full-sample mean return from A3, and drops XOM to
exactly 0%. A single noisy input, the sample mean, has driven half the book into one
name. Min variance instead concentrates in JNJ (50.29%). That makes sense: JNJ has by
far the lowest volatility of the five assets (A3: 18.21% vs. 25 to 29% for the others).
It still gives XOM a real allocation (9.63%), purely for its diversification value, not
its own low volatility.

Comparing weight to risk share makes the difference between the strategies sharp. For
risk parity, risk contributions are exactly 20.00% across all five assets, as they must
be by construction. For the 1/N benchmark, they are not: JPM and XOM together hold 40%
of capital but contribute 50.47% of risk (25.98% plus 24.49%), while JNJ and 7203.T
hold the other 40% of capital but contribute only 26.37% of risk. That is a real
weight/risk gap, though smaller than Lecture 6's own worked example, where a single
stock at roughly 17% of capital carried over 30% of risk. This is some evidence that
our four-sector-plus-currency diversification from A1 makes the naive equal-weight book
somewhat less lopsided than that example, though it does not eliminate the effect.

One property is worth noting explicitly. For min variance, the risk-contribution table
is numerically identical to the weights table (both show 9.45%, 2.47%, 9.63%, 50.29%,
28.16%). This is not a coincidence or a display error. At the interior optimum of a
minimum-variance solve, every included asset has equal marginal risk contribution by
the first-order condition. That mechanically forces each asset's risk-contribution
share to equal its capital-weight share exactly. Min variance and risk parity therefore
agree on what they are claiming, equalise something about risk, but they equalise a
different quantity: marginal risk for min variance, versus total risk contribution for
risk parity. Only the latter is genuinely "equal risk" in the sense A3 and Lecture 6 use
the term.

### In-sample vs. out-of-sample: the real subject of this option

We split the sample chronologically at 70/30. We estimate mu and Sigma on the first 70%
of the data only, solve all three portfolios there, then hold those weights fixed and
apply them to the remaining 30%, never re-estimated. This is the walk-forward
discipline Lectures 2 and 3 insist on. An in-sample number is what the optimiser
promised itself. The out-of-sample number is what an investor actually would have
earned. The 1/N benchmark needs no such split. It estimates nothing, so there is
nothing for it to overfit. We still slice `benchmark_returns` to the identical
out-of-sample window for a fair comparison.

One detail worth flagging directly: we annualise the in-sample estimation with its own,
in-sample-only trading-day count, not A2 Step 7's full-sample `actual_ppy`. Using the
full-sample figure here would annualise the in-sample portfolio construction with a
constant derived partly from out-of-sample dates, a mild look-ahead. The in-sample
estimation shouldn't know anything about the date range of data that comes after the
split. The numerical difference is tiny, since trading-day frequency is very stable
year to year, but the principle is the same one A2 already applies elsewhere. The
out-of-sample evaluation itself still uses the full-sample `actual_ppy`, which is fine,
since that's a hindsight report computed after all the data already exists, not a
real-time decision.

In [ ]:
SPLIT_FRACTION = 0.70
returns_clean = asset_returns_final[TICKERS].dropna()
split_idx = int(len(returns_clean) * SPLIT_FRACTION)
in_sample_dates = returns_clean.index[:split_idx]
out_sample_dates = returns_clean.index[split_idx:]

print(f"In-sample:     {in_sample_dates[0].date()} to {in_sample_dates[-1].date()} "
      f"({len(in_sample_dates)} obs)")
print(f"Out-of-sample: {out_sample_dates[0].date()} to {out_sample_dates[-1].date()} "
      f"({len(out_sample_dates)} obs)")

in_sample_returns = returns_clean.loc[in_sample_dates]
out_sample_returns = returns_clean.loc[out_sample_dates]

actual_ppy_in_sample = len(in_sample_dates) / ((in_sample_dates[-1] - in_sample_dates[0]).days / 365.25)
print(f"In-sample-only periods/year: {actual_ppy_in_sample:.2f} "
      f"(vs. full-sample actual_ppy = {actual_ppy:.2f} used only for the OOS report below)")

mu_is = in_sample_returns.mean() * actual_ppy_in_sample
cov_is = estimate_covariance(in_sample_returns, "ledoit_wolf", periods_per_year=actual_ppy_in_sample)

rf_annual_is = rf_backward_actual.reindex(in_sample_dates).mean() * actual_ppy_in_sample
print(f"Average in-sample annualised risk-free rate: {rf_annual_is:.2%}")

oos_strategies = {}
is_claimed_sharpe = {}
for name, objective in [("Min variance", "min_variance"), ("Max Sharpe", "max_sharpe"),
                        ("Risk parity", "risk_parity")]:
    w, res = optimize_portfolio(mu_is, cov_is, objective=objective, rf=rf_annual_is)
    w_v = w.values
    vol_is = np.sqrt(w_v @ cov_is.values @ w_v)
    is_claimed_sharpe[name] = (w_v @ mu_is.values - rf_annual_is) / vol_is
    oos_strategies[name] = pd.Series(out_sample_returns.values @ w_v, index=out_sample_dates,
                                     name=name)

oos_benchmark = benchmark_returns.reindex(out_sample_dates).dropna()
oos_strategies["1/N benchmark"] = oos_benchmark

oos_rows = {name: compute_stats(r, rf=rf_backward_actual, periods_per_year=actual_ppy)
            for name, r in oos_strategies.items()}
oos_stats = pd.DataFrame(oos_rows).T
oos_stats["in_sample_claimed_sharpe"] = pd.Series(is_claimed_sharpe)

display(oos_stats[["ann_return", "ann_vol", "sharpe", "max_drawdown",
                   "in_sample_claimed_sharpe"]].rename(columns={
    "ann_return": "OOS ann. return", "ann_vol": "OOS ann. vol",
    "sharpe": "OOS Sharpe", "max_drawdown": "OOS max drawdown",
}).style.format({
    "OOS ann. return": "{:.2%}", "OOS ann. vol": "{:.2%}", "OOS Sharpe": "{:.3f}",
    "OOS max drawdown": "{:.2%}", "in_sample_claimed_sharpe": "{:.3f}",
}, na_rep="--"))

**Reading the gap.** Every optimised strategy's claimed in-sample Sharpe exceeds its
realised out-of-sample Sharpe, and by a lot: Min variance 0.918 to 0.091, a 90%
relative collapse; Max Sharpe 1.232 to 0.472, 62%; Risk parity 0.987 to 0.550, 44%. The
1/N benchmark, which estimated nothing, realised a Sharpe of 0.644 out of sample,
higher than all three optimised portfolios. This is exactly the DeMiguel et al. (2009)
finding this course has cited from Lecture 3 onward, now reproduced directly in our own
five-asset universe rather than merely cited from the literature. Per your own
instruction, we report this plainly rather than adjust the sample period until the
answer improves.

The mechanism is more specific than "estimation error" as a single blanket
explanation, though, and it's worth separating out for you rather than reciting the
textbook story uncritically. Max Sharpe fits the standard Chopra-Ziemba account well.
It leaned on the noisiest input, mu, concentrated half the book in AAPL on the strength
of an in-sample mean that would not fully repeat, and paid for it. Risk parity uses no
return information and avoids concentration in either direction. It had the smallest
relative collapse of the three, and the best out-of-sample Sharpe among the optimised
portfolios (0.550). That's consistent with the idea that avoiding concentration,
whatever is driving it, is itself a source of robustness.

Min variance is the genuinely interesting case, and it does not fit the "mu is the
problem" story at all, since it never uses mu. Its realised out-of-sample volatility
(13.43%) was in fact the lowest of all four strategies, including 1/N (14.34%). Min
variance did exactly what it promised on risk. Its Sharpe collapsed because it had the
lowest out-of-sample return (5.51%). That's a direct consequence of concentrating half
the portfolio in JNJ on the strength of in-sample volatility. Volatility is a more
stable, forecastable quantity than returns (Lecture 6), but that still left the
realised portfolio fully exposed to JNJ's own return over 2022 to 2024 specifically, a
period the min-variance objective has no mechanism to say anything about, because it
was never asked to. The lesson generalises: controlling risk well and delivering a good
risk-adjusted return are not the same achievement. A portfolio construction method
should be judged against the objective it actually optimises, not against Sharpe by
default if Sharpe was never its target.

## Extension 2: Tail risk and stress testing (Category A, option A3)

We apply extreme value theory to the same portfolio return series A3's historical and
parametric VaR/ES were computed on (`portfolio_returns`), so the three methods answer
the identical question and can be compared directly. We then stress-test the four
portfolios built in Extension 1 (min variance, max Sharpe, risk parity, 1/N) against
three scenarios, closing the loop this Part B pairing was designed around.

### Threshold selection

Peaks-over-threshold requires choosing a threshold `u`. Too low, and the generalised
Pareto approximation to the tail is invalid, since you'd be fitting the bulk of the
distribution, not the tail. Too high, and too few exceedances remain to fit anything
reliably. We use two standard diagnostics, both computed on the loss series
(`-portfolio_returns`, so exceedances are large losses):

1. **Mean excess plot.** For a true GPD tail, the mean excess function e(u) = E[L - u |
   L > u] is approximately linear in u beyond the point where the GPD approximation
   becomes valid. We look for where the plot stops being erratic and starts tracking a
   straight line.
2. **Parameter stability plot.** We fit the GPD shape parameter xi across a range of
   candidate thresholds and look for a stable plateau. Estimates that swing wildly as
   the threshold moves are a sign there are too few exceedances to trust.

In [ ]:
from scipy.stats import genpareto

portfolio_losses = -portfolio_returns.dropna()

def mean_excess_function(losses, thresholds):
    rows = []
    for u in thresholds:
        exceed = losses[losses > u] - u
        rows.append({"threshold": u,
                     "mean_excess": exceed.mean() if len(exceed) > 10 else np.nan,
                     "n_exceedances": len(exceed)})
    return pd.DataFrame(rows)

candidate_thresholds = np.quantile(portfolio_losses, np.linspace(0.85, 0.99, 30))
mef = mean_excess_function(portfolio_losses, candidate_thresholds)

param_stability = []
for u in candidate_thresholds:
    exceed = portfolio_losses[portfolio_losses > u] - u
    if len(exceed) > 20:
        xi, _, sigma = genpareto.fit(exceed, floc=0)
        param_stability.append({"threshold": u, "shape_xi": xi, "n_exceedances": len(exceed)})
param_stability = pd.DataFrame(param_stability)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(mef["threshold"] * 100, mef["mean_excess"] * 100, "o-", color="#123F69", markersize=3)
axes[0].set_xlabel("threshold u (loss, %)")
axes[0].set_ylabel("mean excess e(u) (%)")
axes[0].set_title("Mean excess plot")

axes[1].plot(param_stability["threshold"] * 100, param_stability["shape_xi"], "o-",
            color="#C0392B", markersize=3)
axes[1].axhline(0, color="grey", linestyle=":", linewidth=1)
axes[1].set_xlabel("threshold u (loss, %)")
axes[1].set_ylabel("fitted shape parameter xi")
axes[1].set_title("Parameter stability plot")

plt.tight_layout()
plt.show()

**Threshold choice.** Neither diagnostic is a textbook-clean single straight line or
plateau, and we say so rather than overstate the clarity of the picture. The mean
excess plot rises gently and somewhat erratically from roughly 0.81% to 1.3%, then
shows a visible kink upward around a 1.7% to 2.0% threshold, before climbing steeply to
the highest candidate thresholds. The parameter stability plot tells a consistent
story: xi drifts upward from roughly 0.13 to 0.17 in the low-threshold region, spikes
locally around the same 1.7% to 2.0% kink, then swings wildly, from 0.28 up to 0.59 and
back down to 0.23, at the highest thresholds, where exceedance counts get very small.

That kink is a genuine ambiguity. It could mean the true GPD region only starts past
2.0%, which the mean-excess theory would read as evidence our chosen threshold is too
low. We keep `CHOSEN_THRESHOLD_QUANTILE = 0.95` (a 1.51% loss threshold, 117
exceedances) anyway, for a specific reason: the region beyond the kink has too few
exceedances to fit reliably, and the wild xi swings past 2% are the direct symptom of
that. Raising the threshold trades a possible bias from too low a threshold for a
definite, visibly larger variance from too few exceedances. The 95% quantile sits in
the calmer part of both diagnostics, after the noisiest low-threshold region but before
the clearly unstable high-threshold region. That's a defensible middle ground given the
trade-off, not a threshold we chose because the theory cleanly pointed at exactly this
value. The comparison against historical and parametric ES below is the real test of
whether this choice was reasonable. If EVT numbers land close to the historical
figures, which have no threshold-choice sensitivity at all, that is evidence our
threshold was adequate.

In [ ]:
CHOSEN_THRESHOLD_QUANTILE = 0.95

def analyze_tail_risk(returns, threshold_quantile=CHOSEN_THRESHOLD_QUANTILE,
                      confidence_levels=(0.95, 0.99)):
    losses = -pd.Series(returns).dropna()
    threshold = losses.quantile(threshold_quantile)
    n = len(losses)
    exceedances = losses[losses > threshold] - threshold
    n_u = len(exceedances)
    xi, _, sigma = genpareto.fit(exceedances, floc=0)

    rows = []
    for alpha in confidence_levels:
        var = threshold + (sigma / xi) * (((n / n_u) * (1 - alpha)) ** (-xi) - 1)
        es = var / (1 - xi) + (sigma - xi * threshold) / (1 - xi) if xi < 1 else np.nan
        rows.append({"confidence": alpha, "EVT_VaR": var, "EVT_ES": es})

    return {
        "threshold": threshold, "threshold_quantile": threshold_quantile,
        "n_exceedances": n_u, "shape_xi": xi, "scale_sigma": sigma,
        "var_es": pd.DataFrame(rows).set_index("confidence"),
    }

tail_result = analyze_tail_risk(portfolio_returns)
print(f"Threshold: {tail_result['threshold']:.2%} loss (quantile {tail_result['threshold_quantile']:.0%}), "
      f"{tail_result['n_exceedances']} exceedances")
print(f"Fitted GPD: shape (xi) = {tail_result['shape_xi']:.4f}, scale (sigma) = {tail_result['scale_sigma']:.4f}")
if tail_result["shape_xi"] > 0:
    print("xi > 0: heavy-tailed (Frechet-type) -- consistent with the excess kurtosis found throughout A3.")
display(tail_result["var_es"].style.format("{:.2%}"))

### Comparing EVT against A3's historical and parametric VaR/ES

In [ ]:
portfolio_hist = var_es[(var_es["asset"] == "Portfolio (1/N)") & (var_es["method"] == "historical")]
portfolio_param = var_es[(var_es["asset"] == "Portfolio (1/N)") & (var_es["method"] == "parametric")]

comparison_rows = []
for alpha in (0.95, 0.99):
    hist_row = portfolio_hist[portfolio_hist["confidence"] == f"{alpha:.0%}"].iloc[0]
    param_row = portfolio_param[portfolio_param["confidence"] == f"{alpha:.0%}"].iloc[0]
    comparison_rows.append({
        "confidence": f"{alpha:.0%}",
        "Historical ES": hist_row["ES"], "Parametric ES": param_row["ES"],
        "EVT ES": tail_result["var_es"].loc[alpha, "EVT_ES"],
    })
method_comparison = pd.DataFrame(comparison_rows).set_index("confidence")
method_comparison["EVT vs Historical"] = method_comparison["EVT ES"] - method_comparison["Historical ES"]
method_comparison["EVT vs Parametric"] = method_comparison["EVT ES"] - method_comparison["Parametric ES"]

display(method_comparison.style.format("{:.2%}"))

**Which understates the tail, and by how much?** Unambiguously the parametric,
Gaussian, method. This is now confirmed by two independent methods agreeing with each
other, rather than just one flagging the other as suspect. EVT ES and historical ES are
close at both confidence levels: 2.44% vs. 2.44% at 95%, identical to two decimal
places, and 4.33% vs. 4.28% at 99%, a 0.05pp gap, which is noise-level given only 117
exceedances feed the GPD fit. Parametric ES sits well below both at every confidence
level: 2.08% at 95%, a 0.36pp shortfall against the other two, and 2.71% at 99%, a
1.62pp shortfall. That means parametric ES99 captures only about 63% of the tail risk
that both the historical and EVT methods, arrived at independently, agree is actually
there (2.71% divided by 4.33%).

This also validates the threshold choice from the previous cell after the fact. Had 117
exceedances and a 95% threshold been a badly wrong choice, EVT would likely have
diverged sharply from the threshold-free historical estimate, rather than landing
within a rounding error of it. The confidence-level pattern matters too. The parametric
gap roughly quadruples from 95% to 99%, the same pattern already seen in A3's own
historical-vs-parametric comparison. That's consistent with a Gaussian distribution
being wrong in a way that gets systematically worse the further into the tail you go,
not a constant, correctable offset. The practical conclusion is the same one A3
reached, now reinforced by a third, independent, non-empirical method: a risk
committee working from the parametric figure at 99% would be short-capitalised for the
tail event by more than a third.

### Tail dependence between assets

Full-sample correlation (A3) measures average co-movement. Tail dependence measures
specifically whether assets crash together, which is the quantity that actually
matters for portfolio risk in a crisis, and the one A3's rolling-correlation finding
(0.21 full-sample average vs. roughly 0.70 during the COVID drawdown) already suggested
is not well summarised by an average correlation alone.

One definition to keep in mind while reading the heatmap below: our coefficient asks,
among the days one asset is in its own worst 5% share, what share of those are also
among the other asset's worst 5% share. A value of 1.0 means every joint-worst day for
one asset is also joint-worst for the other. Independence implies a value of exactly
0.05, the quantile itself, so anything above that is evidence of genuine tail
linkage.

In [ ]:
def tail_dependence(x, y, q=0.05):
    rx, ry = pd.Series(x).rank(pct=True), pd.Series(y).rank(pct=True)
    both = ((rx <= q) & (ry <= q)).sum()
    only_x = (rx <= q).sum()
    return both / only_x if only_x > 0 else np.nan

def tail_dependence_matrix(returns_df, q=0.05):
    cols = returns_df.columns
    mat = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for a in cols:
        for b in cols:
            mat.loc[a, b] = 1.0 if a == b else tail_dependence(returns_df[a], returns_df[b], q=q)
    return mat

TAIL_Q = 0.05
td_matrix = tail_dependence_matrix(asset_returns_final[TICKERS].dropna(), q=TAIL_Q)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(td_matrix.astype(float), annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=1,
           square=True, ax=ax, cbar_kws={"label": f"P(also in worst {TAIL_Q:.0%} | worst {TAIL_Q:.0%})"})
ax.set_title(f"Lower-tail dependence (q={TAIL_Q:.0%})")
plt.tight_layout()
plt.show()

print(f"For reference, independence implies a coefficient of exactly {TAIL_Q:.2f}.")
print("Full-sample correlation matrix, for comparison:")
display(corr_matrix.style.format("{:.3f}"))

**Does full-sample correlation understate crisis-time co-movement?** The honest
answer is: partially, and not in the way we first expected. For most pairs, the
5%-quantile tail dependence coefficient is actually lower than the full-sample
correlation. JPM-XOM, the most correlated pair overall (0.549), shows tail dependence
of 0.35; AAPL-JPM goes from 0.408 to 0.30. That is the opposite direction from a naive
reading of A3's COVID finding. It makes sense once you see the two measures as
different questions. Tail dependence at q=5% asks whether X's worst days and Y's worst
days coincide across the whole sample, scattered over many separate episodes: 2018,
COVID, 2022, the Aug 2024 carry-trade unwind, and more. A3's rolling correlation, by
contrast, measured co-movement specifically during one dated, globally-synchronised
crisis. A diffuse, multi-episode measure and a single-crisis measure need not agree,
and here they do not.

Toyota is the more interesting case, and it only partially fits the original
hypothesis. 7203.T's tail dependence with AAPL (0.09) and XOM (0.15) sits close to or
below its own full-sample correlation (0.108, 0.143), little evidence of hidden
joint-crash risk there. But against JPM (0.19 vs. correlation 0.159), and especially
JNJ (0.10 vs. correlation 0.064, a 56% relative increase and the largest proportional
gap in the entire matrix), tail dependence clearly exceeds full-sample correlation.
Every single coefficient in the matrix exceeds the independence baseline of 0.05,
including the weakest pair (AAPL-7203.T at 0.09). So there is no genuinely independent
pair in this universe, even in the tail. But the degree of tail linkage does not simply
track the degree of average linkage. The practical lesson for you: a full-sample
correlation matrix is a reasonable rough guide to diversification, but not a reliable
one for the specific pairs that matter most in a crisis. Checking tail dependence
directly, rather than assuming it scales with correlation, is worth the extra step.

### Stress testing the Extension 1 portfolios

We apply three scenarios to all four portfolios from Extension 1: min variance, max
Sharpe, risk parity, and 1/N. That closes the loop this Part B pairing was designed
around: does the portfolio construction choice actually change how badly a given shock
hurts? Every scenario reports its result as "portfolio_loss," a positive number, using
the same sign convention as every VaR/ES figure elsewhere in this notebook.

In [ ]:
def perform_stress_test(weights, scenario, asset_data=None, reference_shock=None, target_loss=None):
    w = pd.Series(weights)
    if scenario == "historical":
        cumulative = ((1 + asset_data).prod() - 1).reindex(w.index)
        loss = -(w.values @ cumulative.values)
        return {"scenario": "historical", "portfolio_loss": loss, "asset_moves": cumulative}
    elif scenario == "hypothetical":
        shocks = asset_data.reindex(w.index)
        loss = -(w.values @ shocks.values)
        return {"scenario": "hypothetical", "portfolio_loss": loss, "asset_moves": shocks}
    elif scenario == "reverse":
        ref = reference_shock.reindex(w.index)
        k = -target_loss / (w.values @ ref.values)
        realised = k * ref
        loss = -(w.values @ realised.values)
        return {"scenario": "reverse", "scale_factor": k, "asset_moves": realised,
               "portfolio_loss": loss}
    raise ValueError(scenario)

PORTFOLIOS = {
    "Min variance": w_minvar, "Max Sharpe": w_maxsharpe,
    "Risk parity": w_riskparity, "1/N": w_equal,
}

#### Scenario 1 (historical): replay the COVID crash

We reuse the exact peak/trough window identified in A3's drawdown analysis
(`portfolio_dd_info`). We compound the actual per-asset returns over that window and
apply them to each portfolio's weights.

In [ ]:
covid_window_returns = asset_returns_final[TICKERS].loc[
    portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]
]

historical_results = {name: perform_stress_test(w, "historical", asset_data=covid_window_returns)
                      for name, w in PORTFOLIOS.items()}

historical_summary = pd.Series({name: r["portfolio_loss"] for name, r in historical_results.items()},
                               name="Portfolio loss")
print(f"Historical scenario window: {portfolio_dd_info['peak_date'].date()} to "
      f"{portfolio_dd_info['trough_date'].date()}")
display(historical_summary.to_frame().style.format("{:.2%}"))

In [ ]:
print("Per-asset cumulative return, COVID peak-to-trough window:")
display(historical_results["1/N"]["asset_moves"].to_frame("cumulative return").style.format("{:.2%}"))

#### Scenario 2 (hypothetical): a global trade-tariff shock

This scenario is not drawn from history. It's a narrative one: broad new tariffs and an
escalating trade dispute, with per-asset shocks assigned by economic reasoning about
each firm's actual exposure, not calibrated to hit a target number:

| Asset | Shock | Reasoning |
|---|---|---|
| AAPL | -18% | heavy Asia-based supply chain, consumer demand hit |
| JPM | -12% | credit losses and market-wide stress |
| XOM | +10% | energy security premium, oil price spike on trade disruption |
| JNJ | -5% | defensive sector, mild direct exposure |
| 7203.T | see below | direct tariff target (autos), both an equity and an FX effect |

Toyota gets two separate shocks, not one blended number, deliberately mirroring A2's
currency-conversion lesson: a **-15% local (JPY) equity move** (direct tariff impact on
auto exports) combined with a **+6% JPY appreciation** against USD (safe-haven flows
during trade stress). The FX move partially offsets the equity loss once converted to
USD, exactly the mechanism A2 Step 4 introduced.

In [ ]:
toyota_local_shock = -0.15
jpy_appreciation = 0.06
toyota_usd_shock = (1 + toyota_local_shock) * (1 + jpy_appreciation) - 1
print(f"Toyota: {toyota_local_shock:+.0%} local equity, {jpy_appreciation:+.0%} JPY vs USD "
      f"-> net USD shock {toyota_usd_shock:+.2%}")

hypothetical_shock = pd.Series({
    "AAPL": -0.18, "JPM": -0.12, "XOM": 0.10, "JNJ": -0.05, "7203.T": toyota_usd_shock,
})

hypothetical_results = {name: perform_stress_test(w, "hypothetical", asset_data=hypothetical_shock)
                        for name, w in PORTFOLIOS.items()}
hypothetical_summary = pd.Series({name: r["portfolio_loss"] for name, r in hypothetical_results.items()},
                                 name="Portfolio loss")
display(hypothetical_summary.to_frame().style.format("{:.2%}"))

#### Scenario 3 (reverse): what shock size loses 25%?

Rather than assume a shock and read off the loss, we fix the loss and solve for the
shock. The reference shock shape is the actual per-asset return pattern on the single
worst day in `portfolio_returns`' history, a real, correlation-consistent pattern
rather than an arbitrary vector, scaled up or down until it produces exactly a 25%
portfolio loss. A portfolio that needs a smaller scale factor to reach the same loss is
the more fragile one.

In [ ]:
worst_day = portfolio_returns.idxmin()
reference_shock = asset_returns_final.loc[worst_day, TICKERS]
print(f"Worst single portfolio day: {worst_day.date()}, portfolio return {portfolio_returns.loc[worst_day]:.2%}")
print("Per-asset returns that day (the reference shock shape):")
display(reference_shock.to_frame("return").style.format("{:.2%}"))

TARGET_LOSS = 0.25
reverse_results = {name: perform_stress_test(w, "reverse", reference_shock=reference_shock,
                                              target_loss=TARGET_LOSS)
                   for name, w in PORTFOLIOS.items()}
reverse_summary = pd.DataFrame({
    name: {"Scale factor needed": r["scale_factor"], "Loss achieved (check)": r["portfolio_loss"]}
    for name, r in reverse_results.items()
}).T
display(reverse_summary.style.format({"Scale factor needed": "{:.2f}x", "Loss achieved (check)": "{:.2%}"}))

### Extension 2 discussion

**Which portfolio is most fragile is scenario-dependent, and that is the real
finding.** In the historical COVID replay, Min variance loses least (27.20%), followed
by Max Sharpe (30.32%), Risk parity (32.00%), and 1/N loses most (34.15%). In the
hypothetical trade-tariff scenario the ranking inverts almost completely: Max Sharpe
loses 14.58%, more than double every other portfolio (Min variance 6.34%, Risk parity
7.00%, 1/N 6.98%). The reverse stress test agrees with the hypothetical scenario, not
the historical one. Max Sharpe needs only a 2.21x scale of the reference shock to lose
25%, the smallest multiple of the four (Min variance needs 3.92x, Risk parity 2.93x,
1/N 2.66x). Max Sharpe is the most fragile portfolio to that specific shock shape.

The mechanism is visible directly in Extension 1's weights table. Max Sharpe holds
53.39% AAPL and 22.60% JPM combined, 75.99% of the book in two names. The reverse
stress test's reference shock is the actual 16 March 2020 return pattern, which hit
JPM hardest that single day (-14.96%) and AAPL second-hardest (-12.86%). The
hypothetical tariff scenario separately assigns AAPL its largest shock of any asset
(-18%) plus JPM a material one (-12%). Both scenarios happen to concentrate their
damage on exactly the two names Max Sharpe is overweight. That's not a coincidence:
both scenarios, one from history, one from economic reasoning, were built around
exactly the kind of broad growth/financial-sector stress AAPL and JPM are both exposed
to.

The historical full-window result looks different because it compounds returns over
the whole roughly two-month COVID drawdown, not one day. The per-asset evidence
confirms exactly the mechanism hypothesised above: XOM lost -53.47% over the full
window, easily the worst of the five assets, well past JPM's -42.81%, AAPL's -29.44%,
JNJ's -25.02%, and 7203.T's -20.00%. Toyota held up best of all, plausibly helped by
JPY safe-haven appreciation partially offsetting the local equity decline, the same
mechanism built into the hypothetical scenario's design. Max Sharpe's 0% XOM weight is
therefore precisely what protects it in the historical replay. The arithmetic checks
out exactly against Extension 1's weights table: 9.45% times 29.44%, plus 2.47% times
42.81%, plus 9.63% times 53.47%, plus 50.29% times 25.02%, plus 28.16% times 20.00%,
equals 27.20%, Min variance's reported loss to the decimal. The equivalent sum for Max
Sharpe's weights gives 30.32%, also exact. So the same portfolio can look comparatively
safe or comparatively fragile purely depending on whether a shock is concentrated in a
single day that hits its large names, reverse stress, hypothetical, or accumulates in a
name it happens to have zero exposure to, the historical window's XOM-driven loss.
Neither reading is "the" true fragility of Max Sharpe. Both are correct, conditional on
a specific shock shape, which is the whole justification for running more than one
stress scenario, rather than treating any single one as definitive.

**Risk parity's equal-risk-contribution property is a full-sample, unconditional
guarantee. It does not obviously translate into being the safest portfolio in any
single stress scenario.** It is not the best performer in any of the three scenarios:
third of four in the historical replay; third of four in the hypothetical, essentially
tied with 1/N's 6.98% loss against its own 7.00%; second-most-robust of four in the
reverse test. But it is also never the worst. That's arguably the more useful property
for something meant to generalise across scenarios you have not specifically
anticipated, rather than optimising against the one you happened to test.

**EVT, historical and tail dependence, read together.** EVT and historical ES agree
closely (previous cell), both well above parametric. So the portfolio-level tail risk
this section measures is real, not an artefact of one method's assumptions. The tail
dependence finding refines rather than contradicts that. It shows the specific pairs
driving joint crash risk, JPM-XOM strongest in absolute terms, 7203.T-JNJ and 7203.T-JPM
the pairs where tail dependence most exceeds what full-sample correlation would
predict, are not uniform across the universe. That's exactly why a single
portfolio-level ES number needs the asset-level stress tests above to be actionable.
Knowing the tail is fat is not the same as knowing which position is responsible for
it.

## Part C: Discussion and honest reporting

**1. The benchmark verdict.** No. None of the three optimised portfolios beat the 1/N
benchmark out of sample, on a risk-adjusted basis or otherwise. Out-of-sample Sharpe
was 0.091 (Min variance), 0.472 (Max Sharpe) and 0.550 (Risk parity), all below the
benchmark's 0.644 (Extension 1). This is a direct, in-our-own-data replication of
DeMiguel et al. (2009), and we report it exactly as it came out. We did not adjust the
sample period, the universe, or the split fraction in search of a result where
optimisation wins. Your brief is explicit that the mark is for the quality of the
evidence, not the sign of the finding.

**2. The cost of carelessness.** A2's correction waterfall took the naive Sharpe ratio
(0.855) to a fully corrected figure (0.800), a net change of -0.055, about 6% relative.
The path there was far from monotone, though: dividend adjustment alone pushed Sharpe
up to 1.018 before three further corrections pulled it back down (A2 discussion).
Compare that to the spread across Part B's out-of-sample Sharpe ratios, computed
throughout on the same, fully corrected data: 0.091 to 0.644, a range of 0.553. That's
roughly ten times the magnitude of A2's net data-correction effect. In this specific
comparison, which portfolio-construction method to use mattered far more than which
data corrections to apply. That isn't a contradiction of this course's "data over
models" thesis, A2's corrections still moved individual statistics by up to 0.163 of
Sharpe along the way, but it is a sign that a return- and covariance-sensitive
optimiser compounds estimation error in a way a simple equal-weight calculation never
exposes. A reader who saw only Part B's tables would have no way of knowing any of
this. Every number there is computed on data that was already corrected, with no
visible trace of what the naive figures would have looked like or how much correcting
them mattered. The two sections answer genuinely different questions, and neither
substitutes for the other.

**3. What did not work.** The risk-parity solver in Extension 1 is the clearest
example, and the most instructive precisely because it did not fail loudly. An early
version bounded portfolio weights to (0, 1) during optimisation. The solver reported
`success = True` and returned weights that were, by coincidence, all equal: each
component had been silently clipped to the same upper bound, because the true
unconstrained solution's scale exceeded 1 for every asset given our data. Equal
weights for a "risk parity" portfolio look entirely plausible on inspection, arguably
more plausible than the correct answer, which is not equal-weighted. The bug was only
caught by computing realised risk contributions directly and checking they were
actually equal, rather than trusting `res.success` or eyeballing the output. The lesson
we take from it: a converged optimiser and a plausible-looking answer are not evidence
of correctness on their own. The only real check is verifying the mathematical
property the method is supposed to deliver, on synthetic data with a known answer,
before trusting it on real data.

**4. Limitations.** Three assumptions in this notebook are most likely to break in
practice. First, every Part B expected-return estimate is the simple historical sample
mean. Chopra and Ziemba's most-damaging-input finding is made concrete in Max Sharpe's
53% AAPL concentration: a different realised future would make that allocation look
arbitrary, not sophisticated. Second, our five-asset universe is small and hand-picked
with the benefit of nine years of hindsight (A2's survivorship discussion). Every
diversification, tail-dependence and stress-test finding here is conditional on AAPL,
JPM, XOM, JNJ and Toyota all having survived and thrived. A point-in-time-correct
universe including firms that failed over the same window would very plausibly show
worse tail risk and weaker diversification than we found. Third, nothing in Part B is
costed. A4 already showed the benchmark's own modest turnover (roughly 23% per year) is
not free, and Max Sharpe's more concentrated weights would almost certainly turn over
more between re-estimations, widening the out-of-sample gap in the benchmark's favour
further, not narrowing it.

**5. What next.** With more time, we would pursue B3 (estimation error and robust
covariance), specifically because it targets our own least-understood result. Min
variance collapsed hardest out of sample (90% relative Sharpe decline) despite
achieving the lowest realised out-of-sample volatility of any strategy, including the
benchmark. Its failure was entirely about the asset it concentrated in (JNJ) having a
mediocre return over 2022 to 2024, not about its risk control failing. Comparing
Ledoit-Wolf against a factor-based or hierarchical covariance estimator, and re-running
the identical in-sample/out-of-sample split, would let us test directly whether that
specific failure mode is a covariance-estimation problem we could have fixed, or a
return-side problem no covariance improvement touches. That is currently an open
question this notebook raises but does not answer.

## Bonus: Alternative data, cryptocurrency (Option 1, SOLID tier)

We add Bitcoin (BTC-USD, via yfinance, the most liquid crypto asset with a clean full
history over our 2016-2025 window) and examine exactly the three things this option
asks for: what round-the-clock trading does to daily alignment, to volatility
estimates, and to correlation structure. We do not force BTC into the rest of the
notebook's portfolios. This is a standalone empirical investigation, attempted only
now that Parts A to C are complete, per your brief.

In [ ]:
btc_raw, btc_adj, btc_dl_log = load_price_panel(["BTC-USD"], PRICE_START, PRICE_END,
                                                 cache_dir=DATA_DIR / "crypto")
btc_native = btc_adj["BTC-USD"]
print("BTC-USD download log:", btc_dl_log)
print(f"Native BTC series: {len(btc_native)} observations, "
      f"{btc_native.index[0].date()} to {btc_native.index[-1].date()}")
print(f"Equity panel for comparison: {len(prices_usd)} observations over the same nominal range")

### (a) Daily alignment

BTC trades every calendar day. Our five equities trade on their exchanges' own
weekday-and-holiday calendars. Any joint analysis has to align BTC onto the equity
calendar, which means "Monday's return" for BTC is not really a one-day return. It's
whatever happened from Friday's close through the whole weekend, compressed into a
single reported observation. We test this directly rather than just assert it: if
true, aligned Monday returns should show materially higher variance than
Tuesday-to-Friday returns, roughly on the order of the square root of 3, given three
calendar days are folded into one.

In [ ]:
btc_aligned = btc_native.reindex(prices_usd.index)
btc_aligned_returns = btc_aligned.pct_change().dropna()

weekday_names = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri"}
by_weekday_std = btc_aligned_returns.groupby(btc_aligned_returns.index.weekday).std()
by_weekday_std.index = by_weekday_std.index.map(weekday_names)

print("Std of equity-calendar-aligned BTC returns, by weekday:")
display(by_weekday_std.to_frame("std dev").style.format("{:.4f}"))

monday_std = by_weekday_std.get("Mon", np.nan)
other_std = by_weekday_std.drop("Mon", errors="ignore").mean()
print(f"\nMonday std / Tue-Fri average std: {monday_std / other_std:.2f}x "
      f"(theoretical expectation for 3 folded days vs 1: sqrt(3) = {np.sqrt(3):.2f}x)")

### (b) Volatility estimates

The same annualisation-factor lesson from A2 Step 7 applies here, in a sharper form.
BTC genuinely trades 365.25 days a year, not roughly 260 like our equity panel (A2's
own `actual_ppy`), and not the textbook 252. Using either equity-derived factor for BTC
would understate its annualised volatility, for exactly the same reason A2 Step 7
corrected the equity panel: you are scaling a real daily statistic by the wrong number
of periods per year.

In [ ]:
btc_native_returns = btc_native.pct_change().dropna()
daily_vol_btc = btc_native_returns.std(ddof=1)

ann_vol_correct = daily_vol_btc * np.sqrt(365.25)
ann_vol_wrong_252 = daily_vol_btc * np.sqrt(252)
ann_vol_wrong_equity = daily_vol_btc * np.sqrt(actual_ppy)

vol_comparison = pd.Series({
    "Correct (365.25 days/year, BTC's own calendar)": ann_vol_correct,
    "Wrong: textbook 252": ann_vol_wrong_252,
    f"Wrong: borrowed equity actual_ppy ({actual_ppy:.1f})": ann_vol_wrong_equity,
})
display(vol_comparison.to_frame("Annualised BTC volatility").style.format("{:.2%}"))

understatement = ann_vol_wrong_252 / ann_vol_correct - 1
print(f"\nUsing 252 instead of 365.25 understates BTC's annualised volatility by {understatement:.1%}.")

### (c) Correlation structure

We add BTC (equity-calendar-aligned, from part (a)) into the full-sample correlation
matrix and the rolling-correlation view from A3, reusing both exactly as built. No new
functions are needed here. The question we care about: does BTC behave like a genuine
diversifier against this equity/FX universe on average, and does that hold up during
the COVID drawdown window the way A3 already showed it did not for Toyota?

In [ ]:
returns_with_btc = asset_returns_final[TICKERS].copy()
returns_with_btc["BTC-USD"] = btc_aligned_returns.reindex(returns_with_btc.index)

corr_matrix_btc = returns_with_btc.corr()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr_matrix_btc, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
           square=True, ax=ax, cbar_kws={"label": "correlation"})
ax.set_title("Full-sample correlation matrix, with BTC-USD added")
plt.tight_layout()
plt.show()

print("BTC-USD's correlation with each existing asset:")
display(corr_matrix_btc["BTC-USD"].drop("BTC-USD").to_frame("correlation").style.format("{:.3f}"))

In [ ]:
avg_corr_btc, pairwise_corr_btc = rolling_avg_pairwise_corr(returns_with_btc, window=ROLLING_CORR_WINDOW)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(avg_corr.index, avg_corr.values, color="#9AA5B1", linewidth=1, label="5-asset universe (A3)")
ax.plot(avg_corr_btc.index, avg_corr_btc.values, color="#123F79", linewidth=1.3,
       label="6-asset universe (with BTC)")
ax.axvspan(portfolio_dd_info["peak_date"], portfolio_dd_info["trough_date"],
          color="black", alpha=0.08, label="COVID max-drawdown window")
ax.set_title(f"Rolling ({ROLLING_CORR_WINDOW}d) average pairwise correlation, with and without BTC")
ax.set_xlabel("date")
ax.set_ylabel("average pairwise correlation")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

btc_pairs = [c for c in pairwise_corr_btc.columns if "BTC-USD" in c]
btc_corr_full_sample = pairwise_corr_btc[btc_pairs].mean(axis=1).mean()
btc_corr_covid_window = pairwise_corr_btc[btc_pairs].mean(axis=1).loc[
    portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]
].mean()
print(f"BTC's average rolling correlation with the 5 equities -- full sample: {btc_corr_full_sample:.3f}, "
      f"during the COVID window: {btc_corr_covid_window:.3f}")

### Bonus discussion

**Daily alignment.** The actual Monday/Tue-Fri ratio was 1.43x (Monday std 0.0556 vs
the 0.0354 to 0.0448 range on Tue-Fri), against a theoretical square-root-of-3 = 1.73x
benchmark. The effect is real and in the right direction: Monday is clearly the most
volatile weekday, consistent with three calendar days of news and price action folding
into one observation. But it is noticeably below the naive prediction. The
square-root-of-3 benchmark assumes Saturday, Sunday and Monday moves are independent
and identically distributed, with the same per-day variance as a weekday move, so their
variances would simply add. The shortfall here suggests that assumption doesn't fully
hold for BTC. Weekend crypto moves are not independent extra "free" volatility layered
on top of Monday's. Some of a weekend sell-off or rally partially mean-reverts or gets
partially priced in before Monday's close. Weekend trading volume and liquidity are
also lower than on a typical weekday, which likely dampens the variance contributed by
Saturday and Sunday individually. So the folding mechanism is confirmed, just weaker
than the crude equal-variance assumption predicts.

**Volatility.** Annualising with the correct 365.25-day crypto calendar gives 69.40%,
versus 57.64% using the wrong 252-day equity convention, an understatement of -16.9%.
Using the equity panel's own actual trading-day count (260.2, from A2 Step 7) instead
of a naive 252 barely changes this (58.58%), because that correction only fixes the
roughly 3% equity-specific weekend/holiday gap, not the fundamentally different fact
that BTC trades on essentially all 365.25 days a year. This is the same
annualisation-factor lesson as A2 Step 7 in principle: always annualise with the
sampling frequency actually observed in the data, not a memorised constant. But the gap
here is structural and predictable, since crypto's calendar has roughly 45% more days
than a 252-day equity year, rather than the smaller, more idiosyncratic
260-versus-252 gap the equity panel showed. Reusing an equity-calibrated annualisation
factor for a 24/7 asset understates its true risk by a wide and systematic margin, not
a rounding error.

**Correlation.** BTC's full-sample correlations with the equities were modest and
mostly positive: AAPL 0.172, JPM 0.151, XOM 0.115, JNJ 0.070, and 7203.T -0.015, the
only negative pairwise correlation anywhere in the full correlation matrix. On a
full-sample basis this does support BTC's reputation as a weak diversifier against this
equity universe, and it's essentially uncorrelated, if not slightly negatively
correlated, with Toyota specifically.

The more interesting result is the crisis-window comparison, and it does not confirm
the hypothesis this section originally set out to check. Toyota's correlation with the
rest of the equity universe spiked during the COVID crash (A3/Extension 2), consistent
with the textbook story that a globally-synchronised shock erodes diversification just
when it's needed most. BTC did the opposite: its average rolling correlation with the
equity universe was 0.067 over the full sample, but fell to 0.026 during the COVID
max-drawdown window, lower, not higher, during the crisis. That is a genuinely
different pattern from Toyota's, and it should be reported as such, rather than forced
into the same narrative. A plausible reading is that BTC in 2020 was still a young,
thinly-institutionalised market trading on its own idiosyncratic dynamics. Its
COVID-era crash on 12 March 2020 was severe but happened on its own timeline, driven
substantially by crypto-specific forced deleveraging and exchange liquidations, rather
than moving in lockstep with equity markets tick-for-tick. By later, more
institutionalised episodes, BTC's correlation with risk assets is generally reported
to have risen. The honest conclusion from this dataset alone is that BTC's low
full-sample correlation with equities did not break down in the same way Toyota's did
during this particular crisis window. If anything it strengthened BTC's apparent
diversification benefit in that window. That's a useful reminder that "a crisis breaks
diversification" is not a universal law to be assumed for every asset pair, but an
empirical claim that needs checking asset-by-asset and crisis-by-crisis. The checking
itself, not the answer we expected going in, is the actual lesson of this bonus
section.

## AI use declaration

**Note before submitting:** this declaration is drafted to accurately reflect how AI
was actually used while building this notebook (Claude, via Claude Code). Review it and
edit anything that does not match your own experience before submitting, particularly
the "how we verified" section and the "one thing it got wrong" example, which should
be in your own words, and the signature, which only you can provide.

Tools used: Claude (Claude Code)

What we used them for (tick all that apply):
- [x] explaining concepts or library documentation
- [x] generating code we then reviewed and edited
- [x] debugging errors
- [x] drafting or editing written discussion
- [ ] other: _______________________________

How we verified the output: We ran the notebook ourselves end to end at every stage,
rather than accepting generated code on trust, and fed the actual outputs back for
interpretation, rather than having discussion text written from assumed results.
Several figures were cross-checked against independently verifiable facts. For
example, the A3 drawdown dates (20 Jan to 23 Mar 2020) match the well-documented
historical COVID market bottom, and the Aug 2024 Toyota shock matches the
BoJ/carry-trade unwind described in this subject's own Lecture 6 slides. Key formulas
(VaR/ES sign conventions, GPD-based EVT estimates, risk-parity contributions) were
unit-tested against synthetic data with known properties before being trusted on real
data. *[TODO: add or amend anything that reflects how you personally checked the work,
e.g. re-deriving a formula by hand, or spot-checking an arithmetic claim in the
discussion text yourself.]*

One thing an AI tool got wrong that we caught: an early version of the risk-parity
optimiser in Extension 1 bounded portfolio weights to (0, 1) during optimisation. It
converged successfully and returned weights that were, by coincidence, all equal,
plausible-looking for a "risk parity" portfolio, but wrong. Every weight had been
silently clipped to the same upper bound, because the true unconstrained solution's
scale exceeded 1 for this data. The bug was only caught by computing realised risk
contributions directly and checking they were actually equal, rather than trusting the
optimiser's own success flag. *[TODO: replace with a different example if you'd rather
cite one you personally noticed or asked about.]*

We confirm that we understand all code and text submitted and can explain any part of
it on request.

Signed (write your names): *[TODO, your names here]*